# Test domain samples — 97 URL / 97 hostname

Chạy các ô từ trên xuống (**Run All**). Kaggle cần bật **Internet**; chỉ dùng CPU.
Notebook chọn một URL gốc cho mỗi hostname,
lấy từ `links_corpus.parquet` (4.394.718 dòng; seed 42).
Có 97 mẫu cũ và 0 mẫu bổ sung.
Không cần clone repo, tải Parquet hay upload code Python.

Dùng snapshot crawler và probe hiện tại: tải trực tiếp URL mẫu, retry,
giới hạn tốc độ, tải HTML/PDF, trích xuất và kiểm tra chất lượng (QC).
Kết quả sẽ khác theo thời điểm và môi trường mạng.
HTTP User-Agent là Mozilla/Chrome; Accept và Accept-Language được đặt theo trình duyệt.
User-Agent không đảm bảo hết 403; kết quả thực tế được giữ nguyên.
Bảng ưu tiên domain lỗi xếp theo số URL hostname trong toàn dataset giảm dần,
không xếp theo số lỗi mẫu. Giữ riêng subdomain, số dòng nguồn chưa loại URL trùng.

- **VAO_DUOC**: URL cuối trả HTTP 2xx, kể cả khi trích xuất/QC thất bại.
- **KHONG_VAO_DUOC**: đã chạy nhưng không lấy được HTTP 2xx của trang đích.
- **CHUA_CHAY**: chưa có checkpoint, không tính là lỗi.

`robots_http_status` là HTTP của robots.txt; `http_status` là HTTP của trang.
Dừng do robots là kết quả của chính sách crawler, không chứng minh website
không mở được bằng trình duyệt. Nhóm ưu tiên có thêm đối chiếu Selenium.
Vài mẫu mỗi hostname chưa đại diện cho toàn bộ URL trên website.


Toàn bộ **97 hostname**, mỗi hostname một URL; giữ bộ mẫu reservoir seed 42 ban đầu.

            Chế độ này tải trực tiếp URL mẫu công khai bằng Requests với header trình duyệt;
            robots không chặn bước kiểm tra truy cập. Kết quả Requests và Selenium được giữ riêng.
            `BROWSER_TEST_MODE` mặc định thử Selenium cho URL Requests lỗi, trang JS, text rỗng hoặc QC lỗi;
            `first_each_domain_and_failures` dùng khi muốn đối chiếu thêm mẫu đầu mỗi hostname.
            HTTP 2xx vẫn cần QC: trang lỗi trả 200 hoặc nội dung quảng cáo không phải bài sạch.


## 1. Cấu hình

Mặc định chạy toàn bộ URL đã nhúng. Có thể đặt `MAX_NEW = 10` để thử trước, rồi đổi
thành `None` và chỉ chạy lại ô crawl để làm tiếp. Checkpoint đã có, kể cả
lỗi, được bỏ qua. Muốn thử lại tất cả URL từ đầu, chạy lại ô cấu hình để
tạo thư mục mới. `RESUME_RUN_DIR` dùng để tiếp tục thư mục output đã có.
Code mới không resume trực tiếp run dùng snapshot cũ: đặt `RESUME_RUN_DIR = None`.
Có thể đặt `RECOVERY_RUN_DIR` trỏ tới run cũ đã giải nén để trích lại raw hợp lệ
bằng parser mới và tải lại URL lỗi trong một run mới.


In [ ]:
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36'

WORKERS = 8
DELAY = 1.0
ATTEMPTS = 3
ROBOTS_ATTEMPTS = 3
ROBOTS_CONNECT_TIMEOUT = 15.0
ROBOTS_READ_TIMEOUT = 30.0
ROBOTS_RETRY_BUDGET = 90.0
ROBOTS_FAILURE_TTL = 30.0
NETWORK_CHECK_DOMAINS = ('www.120ask.com', 'bingli.iiyi.com', 'article.iiyi.com')
BROWSER_TEST_MODE = 'failed_only'  # off / first_each_domain_and_failures / all
BROWSER_USER_AGENT = None  # None: dùng User-Agent thực của Chrome đang chạy
BROWSER_LOAD_TIMEOUT = 45.0
BROWSER_ATTEMPTS = 2
CHROME_BINARY = None       # None: Selenium Manager tìm hoặc tải Chrome phù hợp
PAGE_CONNECT_TIMEOUT = 20.0
PAGE_READ_TIMEOUT = 45.0
PAGE_REQUEST_BUDGET = 180.0
PAGE_ATTEMPTS = 3
MAX_NEW = None           # None: toàn bộ; 10: thêm tối đa 10 URL
RESUME_RUN_DIR = None    # Hoặc đường dẫn thư mục run đã có
RECOVERY_RUN_DIR = None  # Run cũ: tạo run mới, reuse raw 2xx và thử lại các URL lỗi

from pathlib import Path
from datetime import datetime, timezone
import json
import sys
import uuid

if sys.version_info < (3, 11):
    raise RuntimeError('Cần Python 3.11 trở lên.')
if type(WORKERS) is not int or WORKERS < 1:
    raise ValueError('WORKERS phải là số nguyên dương.')
if type(ATTEMPTS) is not int or ATTEMPTS < 1:
    raise ValueError('ATTEMPTS phải là số nguyên dương.')
if isinstance(DELAY, bool) or not isinstance(DELAY, (int, float)) or not 0 <= DELAY < float('inf'):
    raise ValueError('DELAY phải hữu hạn và không âm.')
if MAX_NEW is not None and (type(MAX_NEW) is not int or MAX_NEW < 1):
    raise ValueError('MAX_NEW phải là None hoặc số nguyên dương.')
if RECOVERY_RUN_DIR and RESUME_RUN_DIR:
    raise ValueError('Recovery tạo run mới; không đặt đồng thời RESUME_RUN_DIR.')
OUTPUT_BASE = (Path('/kaggle/working') if Path('/kaggle/working').is_dir()
               else Path('/content') if Path('/content').is_dir()
               else Path.cwd() / 'runs')
RUN_DIR = (Path(RESUME_RUN_DIR).expanduser().resolve() if RESUME_RUN_DIR else
           OUTPUT_BASE / ('domain_probe_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
                          + '_' + uuid.uuid4().hex[:6]))
if RESUME_RUN_DIR and not (RUN_DIR / 'manifest.json').is_file():
    raise ValueError('RESUME_RUN_DIR phải chứa manifest.json của notebook này.')
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Output:', RUN_DIR)

DIRECT_PUBLIC_PROBE = True
NETWORK_CHECK_DOMAINS = ('www.cnkang.com', 'www.120ask.com', 'www.familydoctor.com.cn', 'ask.39.net', 'zysjonline.com', 'www.a-hospital.com', 'suckhoecongdongonline.vn', 'zhongyibaodian.net', 'suckhoedoisong.vn', 'nhathuoclongchau.com.vn', 'www.zydcd.com', 'thanhnien.vn', 'www.wujue.com', 'www.youlai.cn', 'laodong.vn', 'baby.39.net', 'phunusuckhoe.giadinhonline.vn', 'www.vinmec.com', 'bingli.iiyi.com', 'medlatec.vn', 'qihuangzhishu.com', 'jbk.39.net', 'giadinhonline.vn', 'baohaiphong.vn', 'baonghean.vn', 'vietnamnet.vn', 'care.39.net', 'test.pmphai.com', 'baodanang.vn', 'tiemchunglongchau.com.vn', 'fitness.39.net', 'baocantho.com.vn', 'hellobacsi.com', 'woman.39.net', 'food.39.net', 'baoangiang.com.vn', 'fk.39.net', 'vov.vn', 'cancer.39.net', 'khoahocphothong.vn', 'jb39.com', 'youmed.vn', 'vov2.vov.vn', 'nk.39.net', 'phuyen.baodaklak.vn', 'www.msdmanuals.cn', 'heart.39.net', 'article.iiyi.com', 'suckhoeviet.org.vn', 'baoquangtri.vn', 'pmc-ecm-healthblog.beta.pharmacity.io', 'man.39.net', 'nhandan.vn', 'baovinhlong.com.vn', 'gan.39.net', 'tamanhhospital.vn', 'tnb.39.net', 'wei.39.net', 'tuoitre.vn', 'baidianfeng.familydoctor.com.cn', 'www.qdnd.vn', 'www.vietnamplus.vn', 'baogialai.com.vn', 'shen.39.net', 'ek.39.net', 'tienphong.vn', 'thaythuocvietnam.vn', 'baoquangninh.vn', 'bachmai.gov.vn', 'www.baidu.com', 'baochinhphu.vn', 'dantri.com.vn', 'gk.39.net', 'vnexpress.net', 'baolangson.vn', 'www.sggp.org.vn', 'gc.39.net', 'pf.39.net', 'health.people.com.cn', 'baophutho.vn', 'hanoimoi.vn', 'benhviennhitrunguong.gov.vn', 'baotayninh.vn', 'sj.39.net', 'baothanhhoa.vn', 'byby.39.net', 'benhvienvietduc.org', 'www.pharmacity.vn', 'www.familydoctor.cn', 'qy.familydoctor.com.cn', 'v.familydoctor.com.cn', 'ask.familydoctor.com.cn', 'mega.vietnamplus.vn', 'yanglao.familydoctor.com.cn', 'fk.99.com.cn', 'special.vietnamplus.vn', 'ypk.familydoctor.com.cn')


## 2. Cài dependencies

Cài các package crawler cần trong môi trường hiện tại. Phiên bản thực tế
và log pip được lưu trong output. Khi resume, dùng phiên bản đã lưu.


In [ ]:
import subprocess
import importlib.metadata

packages = ['pyarrow', 'requests', 'trafilatura', 'pymupdf', 'tld', 'ipython']
if DIRECT_PUBLIC_PROBE and BROWSER_TEST_MODE != 'off':
    packages.append('selenium')
requirement_file = RUN_DIR / 'notebook_requirements.txt'
install_args = ['-r', str(requirement_file)] if requirement_file.exists() else packages
installation = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', *install_args],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    encoding='utf-8', errors='replace',
)
(RUN_DIR / 'pip_install.log').write_text(installation.stdout or '', encoding='utf-8')
if installation.returncode:
    print(installation.stdout)
    raise RuntimeError('Cài dependencies thất bại; xem pip_install.log.')
versions = {name: importlib.metadata.version(name) for name in packages}
if not requirement_file.exists():
    requirement_file.write_text(''.join(f'{p}=={v}\n' for p, v in versions.items()), encoding='utf-8')
print(json.dumps(versions, indent=2))


## 3. Danh sách mẫu và snapshot crawler

97 mẫu giữ nguyên ID/URL nguồn. Ô code có thể mở để kiểm tra danh sách
và snapshot. Crawler được kiểm tra SHA-256 trước khi import.


In [ ]:
SAMPLES = [{'id': 1826763,
  'url': 'http://www.cnkang.com/myjk/201202/480566.html',
  'domain': 'www.cnkang.com',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 2843944,
  'url': 'https://www.120ask.com/question/38159402.htm',
  'domain': 'www.120ask.com',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 3891869,
  'url': 'https://www.familydoctor.com.cn/tangniaobing/a/201702/1706297.html',
  'domain': 'www.familydoctor.com.cn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 2211756,
  'url': 'https://ask.39.net/question/20931974.html',
  'domain': 'ask.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 4182981,
  'url': 'https://zysjonline.com/articles/misc/16204/',
  'domain': 'zysjonline.com',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 2992495,
  'url': 'https://www.a-hospital.com/w/%E4%B8%8A%E8%85%94%E9%9D%99%E8%84%89%E7%BB%BC%E5%90%88%E5%BE%81',
  'domain': 'www.a-hospital.com',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 459741,
  'url': 'https://suckhoecongdongonline.vn/tu-vi-thang-5-2021-am-lich-cua-12-con-giap-chi-tiet-nhat',
  'domain': 'suckhoecongdongonline.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 4065823,
  'url': 'https://zhongyibaodian.net/a/142698.html',
  'domain': 'zhongyibaodian.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 546923,
  'url': 'https://suckhoedoisong.vn/tai-sao-ngay-nao-cung-can-chong-nang-cho-da-16995354.htm',
  'domain': 'suckhoedoisong.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 268030,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/thong-tin-can-biet-ve-phuong-phap-phau-thuat-lay-duong-ro-luan-nhi-59283.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 2065346,
  'url': 'http://www.zydcd.com/zhongyi/zy20700zh/',
  'domain': 'www.zydcd.com',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 651969,
  'url': 'https://thanhnien.vn/tong-hoi-y-hoc-vn-de-nghi-cho-bac-si-vu-tai-bien-chay-than-duoc-tai-ngoai-185676141.htm',
  'domain': 'thanhnien.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 2040931,
  'url': 'http://www.wujue.com/zyzy/zycd/p/200704/14378.html',
  'domain': 'www.wujue.com',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 4013480,
  'url': 'https://www.youlai.cn/yyk/article/608596.html',
  'domain': 'www.youlai.cn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 154292,
  'url': 'https://laodong.vn/suc-khoe/4-thuc-uong-giai-nhiet-giup-giam-trao-nguoc-axit-1205578.ldo',
  'domain': 'laodong.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 772903,
  'url': 'http://baby.39.net/a/141222/4540983.html',
  'domain': 'baby.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 304343,
  'url': 'https://phunusuckhoe.giadinhonline.vn/nhung-loi-ich-bat-ngo-cua-nam-oi-voi-nguoi-benh-tim-mach-c12a511869.html',
  'domain': 'phunusuckhoe.giadinhonline.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 750899,
  'url': 'https://www.vinmec.com/vie/bai-viet/roi-loan-tieu-tien-o-phu-nu-mang-thai-vi',
  'domain': 'www.vinmec.com',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 2501085,
  'url': 'https://bingli.iiyi.com/show/64998-1.html',
  'domain': 'bingli.iiyi.com',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 176969,
  'url': 'https://medlatec.vn/tin-tuc/bi-mat-cam-xuc-khi-quan-he-co-nguyen-nhan-la-do-dau-s78-n22073',
  'domain': 'medlatec.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 895407,
  'url': 'http://qihuangzhishu.com/764/120.htm',
  'domain': 'qihuangzhishu.com',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 2525727,
  'url': 'https://jbk.39.net/shoushu/xws/',
  'domain': 'jbk.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 108128,
  'url': 'https://giadinhonline.vn/giup-me-bau-ngu-ngon-voi-10-meo-don-gian-d19025.html',
  'domain': 'giadinhonline.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 58905,
  'url': 'https://baohaiphong.vn/tat-ca-benh-nhan-covid-19-cua-hai-duong-deu-da-xuat-vien-353669.html',
  'domain': 'baohaiphong.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 67234,
  'url': 'https://baonghean.vn/an-uong-lam-am-co-the-10014055.html',
  'domain': 'baonghean.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 686659,
  'url': 'https://vietnamnet.vn/cach-dap-mat-na-cham-soc-da-mat-phu-hop-voi-tung-loai-da-2018837.html',
  'domain': 'vietnamnet.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 801009,
  'url': 'http://care.39.net/a/170224/5230598.html',
  'domain': 'care.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 2541313,
  'url': 'https://test.pmphai.com/jeesitede/appdisease/toPcDetail?sessionId=&knowledgeLibPrefix=disease&id=0001AA100000000EO44K',
  'domain': 'test.pmphai.com',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 34798,
  'url': 'https://baodanang.vn/khong-chu-quan-voi-benh-glaucoma-3239403.html',
  'domain': 'baodanang.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 674464,
  'url': 'https://tiemchunglongchau.com.vn/kien-thuc-tiem-chung/vi-khuan-nhuom-soi-la-gi-y-nghia-trong-chan-doan-nhiem-trung',
  'domain': 'tiemchunglongchau.com.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 818281,
  'url': 'http://fitness.39.net/a/180124/6017569.html',
  'domain': 'fitness.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 23544,
  'url': 'https://baocantho.com.vn/tai-sao-chung-ta-e-oc-moi-khi-cao-mong-tay-len-bang-a72145.html',
  'domain': 'baocantho.com.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 125814,
  'url': 'https://hellobacsi.com/nuoi-day-con/nhi-khoa/van-de-nhi-khoa-khac/tre-co-ham-rang-bi-o-vang/',
  'domain': 'hellobacsi.com',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 914722,
  'url': 'http://woman.39.net/a/140915/4472540.html',
  'domain': 'woman.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 844242,
  'url': 'http://food.39.net/a/2010821/1451769.html',
  'domain': 'food.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 13933,
  'url': 'https://baoangiang.com.vn/who-canh-bao-nguy-co-lay-lan-cao-cua-virus-chikungunya-a424815.html',
  'domain': 'baoangiang.com.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 836246,
  'url': 'http://fk.39.net/a/241010/i1trnyn.html',
  'domain': 'fk.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 710247,
  'url': 'https://vov.vn/suc-khoe/nhung-thu-co-the-can-khi-ban-gia-di-post981896.vov',
  'domain': 'vov.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 2507808,
  'url': 'https://cancer.39.net/a/210407/8822699.html',
  'domain': 'cancer.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 140282,
  'url': 'https://khoahocphothong.vn/phat-hien-khang-the-co-the-ngan-ngua-hau-het-cac-chung-hiv-240586.html',
  'domain': 'khoahocphothong.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 2516720,
  'url': 'https://jb39.com/jibing/ZiGongNeiMoZengSheng264317.htm',
  'domain': 'jb39.com',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 760817,
  'url': 'https://youmed.vn/tin-tuc/chi-nha-khoa-va-tam-nuoc-dung-cu-nao-tot-hon/',
  'domain': 'youmed.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 717975,
  'url': 'https://vov2.vov.vn/suc-khoe/phong-va-dieu-tri-benh-viem-bo-mi-mat-24968.vov2',
  'domain': 'vov2.vov.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 871353,
  'url': 'http://nk.39.net/a/150617/4640167.html',
  'domain': 'nk.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 312382,
  'url': 'https://phuyen.baodaklak.vn/95/141195/mit-tinh-huong-ung-tuan-le-truyen-thong-ve-phong-chong-khang-thuoc/',
  'domain': 'phuyen.baodaklak.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 3980742,
  'url': 'https://www.msdmanuals.cn/home/skin-disorders/nail-disorders',
  'domain': 'www.msdmanuals.cn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 860914,
  'url': 'http://heart.39.net/a/190801/7350145.html',
  'domain': 'heart.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 2136074,
  'url': 'https://article.iiyi.com/detail/416303.html',
  'domain': 'article.iiyi.com',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 565289,
  'url': 'https://suckhoeviet.org.vn/cong-dung-tuyet-voi-cua-hoa-dau-biec-voi-suc-khoe-va-nhung-luu-y-khi-su-dung-8483.html',
  'domain': 'suckhoeviet.org.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 88645,
  'url': 'https://baoquangtri.vn/suc-khoe/202207/50-benh-nhan-covid-19-dang-dieu-tri-tai-benh-vien-2202255/',
  'domain': 'baoquangtri.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 319666,
  'url': 'https://pmc-ecm-healthblog.beta.pharmacity.io/hoat-chat-amitriptylin/',
  'domain': 'pmc-ecm-healthblog.beta.pharmacity.io',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 868618,
  'url': 'http://man.39.net/a/220924/8244035.html',
  'domain': 'man.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 202493,
  'url': 'https://nhandan.vn/post-853313.html',
  'domain': 'nhandan.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 97029,
  'url': 'https://baovinhlong.com.vn/xa-hoi/y-te/202511/dung-lieu-an-thuc-pham-moc-nguy-hiem-khon-luong-5310d8b/',
  'domain': 'baovinhlong.com.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 852556,
  'url': 'http://gan.39.net/a/260621/x3wnanf.html',
  'domain': 'gan.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 570069,
  'url': 'https://tamanhhospital.vn/tin-tuc/cuu-song-be-so-sinh-bi-thoat-vi-hoanh-the-tui/',
  'domain': 'tamanhhospital.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 905348,
  'url': 'http://tnb.39.net/a/130607/4186274.html',
  'domain': 'tnb.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 909349,
  'url': 'http://wei.39.net/a/110821/1781775.html',
  'domain': 'wei.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 681131,
  'url': 'https://tuoitre.vn/loan-thi-truong-affiliate-ai-chiu-trach-nhiem-chat-luong-san-pham-20260130000926371.htm',
  'domain': 'tuoitre.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 2475161,
  'url': 'https://baidianfeng.familydoctor.com.cn/a/201511/856792.html',
  'domain': 'baidianfeng.familydoctor.com.cn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 722587,
  'url': 'https://www.qdnd.vn/y-te/tin-tuc/covid-hom-nay-12-5-them-2-439-ca-mac-moi-1-benh-nhan-tu-vong-727956',
  'domain': 'www.qdnd.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 732544,
  'url': 'https://www.vietnamplus.vn/to-chuc-y-te-the-gioi-dua-ra-khuyen-nghi-moi-ve-dich-cum-gia-cam-post942110.vnp',
  'domain': 'www.vietnamplus.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 44335,
  'url': 'https://baogialai.com.vn/gia-lai-huong-ung-ngay-asean-phong-chong-benh-sot-xuat-huyet-post280882.html',
  'domain': 'baogialai.com.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 901143,
  'url': 'http://shen.39.net/a/140114/4326874.html',
  'domain': 'shen.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 812548,
  'url': 'http://ek.39.net/a/20101018/1506617.html',
  'domain': 'ek.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 678742,
  'url': 'https://tienphong.vn/weilaiya-white-truffle-2-nam-lien-tiep-la-san-pham-dau-goi-tot-nhat-nam-tai-elle-beauty-awards-2025-post1735881.tpo',
  'domain': 'tienphong.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 660759,
  'url': 'https://thaythuocvietnam.vn/hen-phe-quan-va-nhung-dieu-khong-the-bo-qua/',
  'domain': 'thaythuocvietnam.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 84125,
  'url': 'https://baoquangninh.vn/tap-huan-cong-tac-giam-sat-phong-chong-dich-benh-do-vi-rut-nipah-niv-3394596.html',
  'domain': 'baoquangninh.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 1143,
  'url': 'https://bachmai.gov.vn/bai-viet/canh-bao-nguoi-lon-cung-dinh-thuy-dau?id=1e8dc2c0-88db-e42e-d5db-cb9137df465f',
  'domain': 'bachmai.gov.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 3160976,
  'url': 'https://www.baidu.com/bh/dict/ydxx_8424738550787796805?from=dicta&sf_ref=med_pc&sf_ch=ch_med_pc',
  'domain': 'www.baidu.com',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 26481,
  'url': 'https://baochinhphu.vn/cac-dia-phuong-can-day-nhanh-tien-do-tiem-chung-vaccine-covid-19-10222060115443633.htm',
  'domain': 'baochinhphu.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 99635,
  'url': 'https://dantri.com.vn/suc-khoe/dung-thuc-pham-chuc-nang-nhu-the-nao-de-khong-loi-bat-cap-hai-ky-4-20251009161639294.htm',
  'domain': 'dantri.com.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 854825,
  'url': 'http://gk.39.net/a/090624/900447.html',
  'domain': 'gk.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 702221,
  'url': 'https://vnexpress.net/bo-y-te-virus-nipah-co-tiem-nang-gay-dai-dich-5041215.html',
  'domain': 'vnexpress.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 64079,
  'url': 'https://baolangson.vn/thanh-tra-kiem-tra-kiem-nghiem-san-pham-duoc-tiep-nhan-xu-ly-cac-thong-tin-ve-thuoc-gia-5044678.html',
  'domain': 'baolangson.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 724965,
  'url': 'https://www.sggp.org.vn/ho-tro-con-dao-thuc-hien-phau-thuat-noi-soi-co-ke-hoach-post817576.html',
  'domain': 'www.sggp.org.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 853240,
  'url': 'http://gc.39.net/a/141221/4540391.html',
  'domain': 'gc.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 876479,
  'url': 'http://pf.39.net/a/2011418/1666222.html',
  'domain': 'pf.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 857643,
  'url': 'http://health.people.com.cn/n1/2026/0701/c14739-40751470.html',
  'domain': 'health.people.com.cn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 80524,
  'url': 'https://baophutho.vn/4-mon-pho-bien-tren-mam-co-cung-ram-thang-gieng-co-y-nghia-mang-lai-may-man-248768.htm',
  'domain': 'baophutho.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 118141,
  'url': 'https://hanoimoi.vn/dung-de-thuc-pham-ngay-tet-thanh-ganh-nang-cho-gan-ruot-734057.html',
  'domain': 'hanoimoi.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 98260,
  'url': 'https://benhviennhitrunguong.gov.vn/thuoc-diet-chuot-va-nguy-co-xuat-huyet-o-tre-em.html',
  'domain': 'benhviennhitrunguong.gov.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 90572,
  'url': 'https://baotayninh.vn/hanh-trinh-tan-hien-cua-nhung-thay-thuoc-uu-tu-140074.html',
  'domain': 'baotayninh.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 904470,
  'url': 'http://sj.39.net/a/140814/4447769.html',
  'domain': 'sj.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 91319,
  'url': 'https://baothanhhoa.vn/bac-sy-benh-vien-108-da-thuc-hien-thanh-cong-300-ca-ghep-gan-267964.htm',
  'domain': 'baothanhhoa.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 794284,
  'url': 'http://byby.39.net/a/230524/13823332.html',
  'domain': 'byby.39.net',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 98535,
  'url': 'https://benhvienvietduc.org/chuyen-gia-benh-vien-huu-nghi-viet-duc-tu-van-benh-ly-di-dang-dong-tinh-mach-nao.html',
  'domain': 'benhvienvietduc.org',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 720463,
  'url': 'https://www.pharmacity.vn/tang-sinh-mon-sau-khi-sinh-lieu-co-nen-khau-ngay-lap-tuc.htm',
  'domain': 'www.pharmacity.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 3530740,
  'url': 'https://www.familydoctor.cn/a/201309/512842.html',
  'domain': 'www.familydoctor.cn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 900894,
  'url': 'http://qy.familydoctor.com.cn/a/201405/609715.html',
  'domain': 'qy.familydoctor.com.cn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 2551069,
  'url': 'https://v.familydoctor.com.cn/a/201509/816823.html',
  'domain': 'v.familydoctor.com.cn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 766444,
  'url': 'http://ask.familydoctor.com.cn/zt/mn.html',
  'domain': 'ask.familydoctor.com.cn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 199047,
  'url': 'https://mega.vietnamplus.vn/thai-huong-nu-anh-hung-dam-me-lam-cho-dat-no-hoa-5415.html',
  'domain': 'mega.vietnamplus.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 2131080,
  'url': 'http://yanglao.familydoctor.com.cn/zt/scsjtfz.html',
  'domain': 'yanglao.familydoctor.com.cn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 836542,
  'url': 'http://fk.99.com.cn/news/410997.html',
  'domain': 'fk.99.com.cn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 583,
  'url': 'http://special.vietnamplus.vn/benh_vien_nhan_ai',
  'domain': 'special.vietnamplus.vn',
  'sample_role': 'original',
  'sample_index': 0},
 {'id': 4030514,
  'url': 'https://ypk.familydoctor.com.cn/267067/',
  'domain': 'ypk.familydoctor.com.cn',
  'sample_role': 'original',
  'sample_index': 0}]

SOURCE_PROFILE = {'rows': 4394718,
 'id_min': 583,
 'id_max': 4420561,
 'duplicate_ids': 0,
 'hosts': 97,
 'host_counts': {'special.vietnamplus.vn': 1,
                 'bachmai.gov.vn': 2783,
                 'baoangiang.com.vn': 10738,
                 'baocantho.com.vn': 12016,
                 'baochinhphu.vn': 2302,
                 'baodanang.vn': 14500,
                 'baogialai.com.vn': 3290,
                 'baohaiphong.vn': 16286,
                 'baolangson.vn': 1981,
                 'baonghean.vn': 15941,
                 'baophutho.vn': 1288,
                 'baoquangninh.vn': 2912,
                 'baoquangtri.vn': 5483,
                 'baotayninh.vn': 1198,
                 'baothanhhoa.vn': 782,
                 'baovinhlong.com.vn': 5113,
                 'benhviennhitrunguong.gov.vn': 1200,
                 'benhvienvietduc.org': 425,
                 'dantri.com.vn': 2063,
                 'giadinhonline.vn': 16729,
                 'hanoimoi.vn': 1283,
                 'hellobacsi.com': 11747,
                 'khoahocphothong.vn': 8143,
                 'laodong.vn': 31282,
                 'medlatec.vn': 24762,
                 'mega.vietnamplus.vn': 2,
                 'nhandan.vn': 5225,
                 'nhathuoclongchau.com.vn': 79598,
                 'phunusuckhoe.giadinhonline.vn': 27577,
                 'phuyen.baodaklak.vn': 6252,
                 'pmc-ecm-healthblog.beta.pharmacity.io': 5333,
                 'suckhoecongdongonline.vn': 154503,
                 'suckhoedoisong.vn': 85823,
                 'suckhoeviet.org.vn': 5627,
                 'tamanhhospital.vn': 4261,
                 'thanhnien.vn': 72111,
                 'thaythuocvietnam.vn': 2927,
                 'tiemchunglongchau.com.vn': 13281,
                 'tienphong.vn': 3021,
                 'tuoitre.vn': 3893,
                 'vietnamnet.vn': 15543,
                 'vnexpress.net': 2055,
                 'vov.vn': 9784,
                 'vov2.vov.vn': 6899,
                 'www.pharmacity.vn': 69,
                 'www.qdnd.vn': 3700,
                 'www.sggp.org.vn': 1853,
                 'www.vietnamplus.vn': 3695,
                 'www.vinmec.com': 26129,
                 'youmed.vn': 7189,
                 'ask.familydoctor.com.cn': 3,
                 'baby.39.net': 27621,
                 'byby.39.net': 439,
                 'cancer.39.net': 8415,
                 'care.39.net': 15134,
                 'ek.39.net': 3222,
                 'fitness.39.net': 12066,
                 'fk.39.net': 9812,
                 'fk.99.com.cn': 1,
                 'food.39.net': 11024,
                 'gan.39.net': 5027,
                 'gc.39.net': 1746,
                 'gk.39.net': 2059,
                 'health.people.com.cn': 1348,
                 'heart.39.net': 5965,
                 'man.39.net': 5320,
                 'nk.39.net': 6650,
                 'pf.39.net': 1538,
                 'qihuangzhishu.com': 23637,
                 'qy.familydoctor.com.cn': 40,
                 'shen.39.net': 3223,
                 'sj.39.net': 942,
                 'tnb.39.net': 4178,
                 'wei.39.net': 3894,
                 'woman.39.net': 11687,
                 'www.120ask.com': 918479,
                 'www.cnkang.com': 963438,
                 'www.wujue.com': 53341,
                 'www.zydcd.com': 78680,
                 'yanglao.familydoctor.com.cn': 2,
                 'article.iiyi.com': 5641,
                 'ask.39.net': 337375,
                 'baidianfeng.familydoctor.com.cn': 3798,
                 'bingli.iiyi.com': 24990,
                 'jb39.com': 7306,
                 'jbk.39.net': 19142,
                 'test.pmphai.com': 15120,
                 'v.familydoctor.com.cn': 10,
                 'www.a-hospital.com': 169339,
                 'www.baidu.com': 2631,
                 'www.familydoctor.cn': 44,
                 'www.familydoctor.com.cn': 447453,
                 'www.msdmanuals.cn': 6173,
                 'www.youlai.cn': 46119,
                 'ypk.familydoctor.com.cn': 1,
                 'zhongyibaodian.net': 146700,
                 'zysjonline.com': 243347},
 'first_1000_host_counts': {'special.vietnamplus.vn': 1, 'bachmai.gov.vn': 999},
 'sample_host_counts': {'article.iiyi.com': 1,
                        'ask.39.net': 1,
                        'ask.familydoctor.com.cn': 1,
                        'baby.39.net': 1,
                        'bachmai.gov.vn': 1,
                        'baidianfeng.familydoctor.com.cn': 1,
                        'baoangiang.com.vn': 1,
                        'baocantho.com.vn': 1,
                        'baochinhphu.vn': 1,
                        'baodanang.vn': 1,
                        'baogialai.com.vn': 1,
                        'baohaiphong.vn': 1,
                        'baolangson.vn': 1,
                        'baonghean.vn': 1,
                        'baophutho.vn': 1,
                        'baoquangninh.vn': 1,
                        'baoquangtri.vn': 1,
                        'baotayninh.vn': 1,
                        'baothanhhoa.vn': 1,
                        'baovinhlong.com.vn': 1,
                        'benhviennhitrunguong.gov.vn': 1,
                        'benhvienvietduc.org': 1,
                        'bingli.iiyi.com': 1,
                        'byby.39.net': 1,
                        'cancer.39.net': 1,
                        'care.39.net': 1,
                        'dantri.com.vn': 1,
                        'ek.39.net': 1,
                        'fitness.39.net': 1,
                        'fk.39.net': 1,
                        'fk.99.com.cn': 1,
                        'food.39.net': 1,
                        'gan.39.net': 1,
                        'gc.39.net': 1,
                        'giadinhonline.vn': 1,
                        'gk.39.net': 1,
                        'hanoimoi.vn': 1,
                        'health.people.com.cn': 1,
                        'heart.39.net': 1,
                        'hellobacsi.com': 1,
                        'jb39.com': 1,
                        'jbk.39.net': 1,
                        'khoahocphothong.vn': 1,
                        'laodong.vn': 1,
                        'man.39.net': 1,
                        'medlatec.vn': 1,
                        'mega.vietnamplus.vn': 1,
                        'nhandan.vn': 1,
                        'nhathuoclongchau.com.vn': 1,
                        'nk.39.net': 1,
                        'pf.39.net': 1,
                        'phunusuckhoe.giadinhonline.vn': 1,
                        'phuyen.baodaklak.vn': 1,
                        'pmc-ecm-healthblog.beta.pharmacity.io': 1,
                        'qihuangzhishu.com': 1,
                        'qy.familydoctor.com.cn': 1,
                        'shen.39.net': 1,
                        'sj.39.net': 1,
                        'special.vietnamplus.vn': 1,
                        'suckhoecongdongonline.vn': 1,
                        'suckhoedoisong.vn': 1,
                        'suckhoeviet.org.vn': 1,
                        'tamanhhospital.vn': 1,
                        'test.pmphai.com': 1,
                        'thanhnien.vn': 1,
                        'thaythuocvietnam.vn': 1,
                        'tiemchunglongchau.com.vn': 1,
                        'tienphong.vn': 1,
                        'tnb.39.net': 1,
                        'tuoitre.vn': 1,
                        'v.familydoctor.com.cn': 1,
                        'vietnamnet.vn': 1,
                        'vnexpress.net': 1,
                        'vov.vn': 1,
                        'vov2.vov.vn': 1,
                        'wei.39.net': 1,
                        'woman.39.net': 1,
                        'www.120ask.com': 1,
                        'www.a-hospital.com': 1,
                        'www.baidu.com': 1,
                        'www.cnkang.com': 1,
                        'www.familydoctor.cn': 1,
                        'www.familydoctor.com.cn': 1,
                        'www.msdmanuals.cn': 1,
                        'www.pharmacity.vn': 1,
                        'www.qdnd.vn': 1,
                        'www.sggp.org.vn': 1,
                        'www.vietnamplus.vn': 1,
                        'www.vinmec.com': 1,
                        'www.wujue.com': 1,
                        'www.youlai.cn': 1,
                        'www.zydcd.com': 1,
                        'yanglao.familydoctor.com.cn': 1,
                        'youmed.vn': 1,
                        'ypk.familydoctor.com.cn': 1,
                        'zhongyibaodian.net': 1,
                        'zysjonline.com': 1}}

BASELINE_DOMAINS = {'www.cnkang.com': {'error_type': 'not_tested',
                    'requested_additional': 0,
                    'selected_additional': 0,
                    'shortfall': 0},
 'www.120ask.com': {'error_type': 'not_tested',
                    'requested_additional': 0,
                    'selected_additional': 0,
                    'shortfall': 0},
 'www.familydoctor.com.cn': {'error_type': 'not_tested',
                             'requested_additional': 0,
                             'selected_additional': 0,
                             'shortfall': 0},
 'ask.39.net': {'error_type': 'not_tested',
                'requested_additional': 0,
                'selected_additional': 0,
                'shortfall': 0},
 'zysjonline.com': {'error_type': 'not_tested',
                    'requested_additional': 0,
                    'selected_additional': 0,
                    'shortfall': 0},
 'www.a-hospital.com': {'error_type': 'not_tested',
                        'requested_additional': 0,
                        'selected_additional': 0,
                        'shortfall': 0},
 'suckhoecongdongonline.vn': {'error_type': 'not_tested',
                              'requested_additional': 0,
                              'selected_additional': 0,
                              'shortfall': 0},
 'zhongyibaodian.net': {'error_type': 'not_tested',
                        'requested_additional': 0,
                        'selected_additional': 0,
                        'shortfall': 0},
 'suckhoedoisong.vn': {'error_type': 'not_tested',
                       'requested_additional': 0,
                       'selected_additional': 0,
                       'shortfall': 0},
 'nhathuoclongchau.com.vn': {'error_type': 'not_tested',
                             'requested_additional': 0,
                             'selected_additional': 0,
                             'shortfall': 0},
 'www.zydcd.com': {'error_type': 'not_tested',
                   'requested_additional': 0,
                   'selected_additional': 0,
                   'shortfall': 0},
 'thanhnien.vn': {'error_type': 'not_tested',
                  'requested_additional': 0,
                  'selected_additional': 0,
                  'shortfall': 0},
 'www.wujue.com': {'error_type': 'not_tested',
                   'requested_additional': 0,
                   'selected_additional': 0,
                   'shortfall': 0},
 'www.youlai.cn': {'error_type': 'not_tested',
                   'requested_additional': 0,
                   'selected_additional': 0,
                   'shortfall': 0},
 'laodong.vn': {'error_type': 'not_tested',
                'requested_additional': 0,
                'selected_additional': 0,
                'shortfall': 0},
 'baby.39.net': {'error_type': 'not_tested',
                 'requested_additional': 0,
                 'selected_additional': 0,
                 'shortfall': 0},
 'phunusuckhoe.giadinhonline.vn': {'error_type': 'not_tested',
                                   'requested_additional': 0,
                                   'selected_additional': 0,
                                   'shortfall': 0},
 'www.vinmec.com': {'error_type': 'not_tested',
                    'requested_additional': 0,
                    'selected_additional': 0,
                    'shortfall': 0},
 'bingli.iiyi.com': {'error_type': 'not_tested',
                     'requested_additional': 0,
                     'selected_additional': 0,
                     'shortfall': 0},
 'medlatec.vn': {'error_type': 'not_tested',
                 'requested_additional': 0,
                 'selected_additional': 0,
                 'shortfall': 0},
 'qihuangzhishu.com': {'error_type': 'not_tested',
                       'requested_additional': 0,
                       'selected_additional': 0,
                       'shortfall': 0},
 'jbk.39.net': {'error_type': 'not_tested',
                'requested_additional': 0,
                'selected_additional': 0,
                'shortfall': 0},
 'giadinhonline.vn': {'error_type': 'not_tested',
                      'requested_additional': 0,
                      'selected_additional': 0,
                      'shortfall': 0},
 'baohaiphong.vn': {'error_type': 'not_tested',
                    'requested_additional': 0,
                    'selected_additional': 0,
                    'shortfall': 0},
 'baonghean.vn': {'error_type': 'not_tested',
                  'requested_additional': 0,
                  'selected_additional': 0,
                  'shortfall': 0},
 'vietnamnet.vn': {'error_type': 'not_tested',
                   'requested_additional': 0,
                   'selected_additional': 0,
                   'shortfall': 0},
 'care.39.net': {'error_type': 'not_tested',
                 'requested_additional': 0,
                 'selected_additional': 0,
                 'shortfall': 0},
 'test.pmphai.com': {'error_type': 'not_tested',
                     'requested_additional': 0,
                     'selected_additional': 0,
                     'shortfall': 0},
 'baodanang.vn': {'error_type': 'not_tested',
                  'requested_additional': 0,
                  'selected_additional': 0,
                  'shortfall': 0},
 'tiemchunglongchau.com.vn': {'error_type': 'not_tested',
                              'requested_additional': 0,
                              'selected_additional': 0,
                              'shortfall': 0},
 'fitness.39.net': {'error_type': 'not_tested',
                    'requested_additional': 0,
                    'selected_additional': 0,
                    'shortfall': 0},
 'baocantho.com.vn': {'error_type': 'not_tested',
                      'requested_additional': 0,
                      'selected_additional': 0,
                      'shortfall': 0},
 'hellobacsi.com': {'error_type': 'not_tested',
                    'requested_additional': 0,
                    'selected_additional': 0,
                    'shortfall': 0},
 'woman.39.net': {'error_type': 'not_tested',
                  'requested_additional': 0,
                  'selected_additional': 0,
                  'shortfall': 0},
 'food.39.net': {'error_type': 'not_tested',
                 'requested_additional': 0,
                 'selected_additional': 0,
                 'shortfall': 0},
 'baoangiang.com.vn': {'error_type': 'not_tested',
                       'requested_additional': 0,
                       'selected_additional': 0,
                       'shortfall': 0},
 'fk.39.net': {'error_type': 'not_tested',
               'requested_additional': 0,
               'selected_additional': 0,
               'shortfall': 0},
 'vov.vn': {'error_type': 'not_tested', 'requested_additional': 0, 'selected_additional': 0, 'shortfall': 0},
 'cancer.39.net': {'error_type': 'not_tested',
                   'requested_additional': 0,
                   'selected_additional': 0,
                   'shortfall': 0},
 'khoahocphothong.vn': {'error_type': 'not_tested',
                        'requested_additional': 0,
                        'selected_additional': 0,
                        'shortfall': 0},
 'jb39.com': {'error_type': 'not_tested',
              'requested_additional': 0,
              'selected_additional': 0,
              'shortfall': 0},
 'youmed.vn': {'error_type': 'not_tested',
               'requested_additional': 0,
               'selected_additional': 0,
               'shortfall': 0},
 'vov2.vov.vn': {'error_type': 'not_tested',
                 'requested_additional': 0,
                 'selected_additional': 0,
                 'shortfall': 0},
 'nk.39.net': {'error_type': 'not_tested',
               'requested_additional': 0,
               'selected_additional': 0,
               'shortfall': 0},
 'phuyen.baodaklak.vn': {'error_type': 'not_tested',
                         'requested_additional': 0,
                         'selected_additional': 0,
                         'shortfall': 0},
 'www.msdmanuals.cn': {'error_type': 'not_tested',
                       'requested_additional': 0,
                       'selected_additional': 0,
                       'shortfall': 0},
 'heart.39.net': {'error_type': 'not_tested',
                  'requested_additional': 0,
                  'selected_additional': 0,
                  'shortfall': 0},
 'article.iiyi.com': {'error_type': 'not_tested',
                      'requested_additional': 0,
                      'selected_additional': 0,
                      'shortfall': 0},
 'suckhoeviet.org.vn': {'error_type': 'not_tested',
                        'requested_additional': 0,
                        'selected_additional': 0,
                        'shortfall': 0},
 'baoquangtri.vn': {'error_type': 'not_tested',
                    'requested_additional': 0,
                    'selected_additional': 0,
                    'shortfall': 0},
 'pmc-ecm-healthblog.beta.pharmacity.io': {'error_type': 'not_tested',
                                           'requested_additional': 0,
                                           'selected_additional': 0,
                                           'shortfall': 0},
 'man.39.net': {'error_type': 'not_tested',
                'requested_additional': 0,
                'selected_additional': 0,
                'shortfall': 0},
 'nhandan.vn': {'error_type': 'not_tested',
                'requested_additional': 0,
                'selected_additional': 0,
                'shortfall': 0},
 'baovinhlong.com.vn': {'error_type': 'not_tested',
                        'requested_additional': 0,
                        'selected_additional': 0,
                        'shortfall': 0},
 'gan.39.net': {'error_type': 'not_tested',
                'requested_additional': 0,
                'selected_additional': 0,
                'shortfall': 0},
 'tamanhhospital.vn': {'error_type': 'not_tested',
                       'requested_additional': 0,
                       'selected_additional': 0,
                       'shortfall': 0},
 'tnb.39.net': {'error_type': 'not_tested',
                'requested_additional': 0,
                'selected_additional': 0,
                'shortfall': 0},
 'wei.39.net': {'error_type': 'not_tested',
                'requested_additional': 0,
                'selected_additional': 0,
                'shortfall': 0},
 'tuoitre.vn': {'error_type': 'not_tested',
                'requested_additional': 0,
                'selected_additional': 0,
                'shortfall': 0},
 'baidianfeng.familydoctor.com.cn': {'error_type': 'not_tested',
                                     'requested_additional': 0,
                                     'selected_additional': 0,
                                     'shortfall': 0},
 'www.qdnd.vn': {'error_type': 'not_tested',
                 'requested_additional': 0,
                 'selected_additional': 0,
                 'shortfall': 0},
 'www.vietnamplus.vn': {'error_type': 'not_tested',
                        'requested_additional': 0,
                        'selected_additional': 0,
                        'shortfall': 0},
 'baogialai.com.vn': {'error_type': 'not_tested',
                      'requested_additional': 0,
                      'selected_additional': 0,
                      'shortfall': 0},
 'shen.39.net': {'error_type': 'not_tested',
                 'requested_additional': 0,
                 'selected_additional': 0,
                 'shortfall': 0},
 'ek.39.net': {'error_type': 'not_tested',
               'requested_additional': 0,
               'selected_additional': 0,
               'shortfall': 0},
 'tienphong.vn': {'error_type': 'not_tested',
                  'requested_additional': 0,
                  'selected_additional': 0,
                  'shortfall': 0},
 'thaythuocvietnam.vn': {'error_type': 'not_tested',
                         'requested_additional': 0,
                         'selected_additional': 0,
                         'shortfall': 0},
 'baoquangninh.vn': {'error_type': 'not_tested',
                     'requested_additional': 0,
                     'selected_additional': 0,
                     'shortfall': 0},
 'bachmai.gov.vn': {'error_type': 'not_tested',
                    'requested_additional': 0,
                    'selected_additional': 0,
                    'shortfall': 0},
 'www.baidu.com': {'error_type': 'not_tested',
                   'requested_additional': 0,
                   'selected_additional': 0,
                   'shortfall': 0},
 'baochinhphu.vn': {'error_type': 'not_tested',
                    'requested_additional': 0,
                    'selected_additional': 0,
                    'shortfall': 0},
 'dantri.com.vn': {'error_type': 'not_tested',
                   'requested_additional': 0,
                   'selected_additional': 0,
                   'shortfall': 0},
 'gk.39.net': {'error_type': 'not_tested',
               'requested_additional': 0,
               'selected_additional': 0,
               'shortfall': 0},
 'vnexpress.net': {'error_type': 'not_tested',
                   'requested_additional': 0,
                   'selected_additional': 0,
                   'shortfall': 0},
 'baolangson.vn': {'error_type': 'not_tested',
                   'requested_additional': 0,
                   'selected_additional': 0,
                   'shortfall': 0},
 'www.sggp.org.vn': {'error_type': 'not_tested',
                     'requested_additional': 0,
                     'selected_additional': 0,
                     'shortfall': 0},
 'gc.39.net': {'error_type': 'not_tested',
               'requested_additional': 0,
               'selected_additional': 0,
               'shortfall': 0},
 'pf.39.net': {'error_type': 'not_tested',
               'requested_additional': 0,
               'selected_additional': 0,
               'shortfall': 0},
 'health.people.com.cn': {'error_type': 'not_tested',
                          'requested_additional': 0,
                          'selected_additional': 0,
                          'shortfall': 0},
 'baophutho.vn': {'error_type': 'not_tested',
                  'requested_additional': 0,
                  'selected_additional': 0,
                  'shortfall': 0},
 'hanoimoi.vn': {'error_type': 'not_tested',
                 'requested_additional': 0,
                 'selected_additional': 0,
                 'shortfall': 0},
 'benhviennhitrunguong.gov.vn': {'error_type': 'not_tested',
                                 'requested_additional': 0,
                                 'selected_additional': 0,
                                 'shortfall': 0},
 'baotayninh.vn': {'error_type': 'not_tested',
                   'requested_additional': 0,
                   'selected_additional': 0,
                   'shortfall': 0},
 'sj.39.net': {'error_type': 'not_tested',
               'requested_additional': 0,
               'selected_additional': 0,
               'shortfall': 0},
 'baothanhhoa.vn': {'error_type': 'not_tested',
                    'requested_additional': 0,
                    'selected_additional': 0,
                    'shortfall': 0},
 'byby.39.net': {'error_type': 'not_tested',
                 'requested_additional': 0,
                 'selected_additional': 0,
                 'shortfall': 0},
 'benhvienvietduc.org': {'error_type': 'not_tested',
                         'requested_additional': 0,
                         'selected_additional': 0,
                         'shortfall': 0},
 'www.pharmacity.vn': {'error_type': 'not_tested',
                       'requested_additional': 0,
                       'selected_additional': 0,
                       'shortfall': 0},
 'www.familydoctor.cn': {'error_type': 'not_tested',
                         'requested_additional': 0,
                         'selected_additional': 0,
                         'shortfall': 0},
 'qy.familydoctor.com.cn': {'error_type': 'not_tested',
                            'requested_additional': 0,
                            'selected_additional': 0,
                            'shortfall': 0},
 'v.familydoctor.com.cn': {'error_type': 'not_tested',
                           'requested_additional': 0,
                           'selected_additional': 0,
                           'shortfall': 0},
 'ask.familydoctor.com.cn': {'error_type': 'not_tested',
                             'requested_additional': 0,
                             'selected_additional': 0,
                             'shortfall': 0},
 'mega.vietnamplus.vn': {'error_type': 'not_tested',
                         'requested_additional': 0,
                         'selected_additional': 0,
                         'shortfall': 0},
 'yanglao.familydoctor.com.cn': {'error_type': 'not_tested',
                                 'requested_additional': 0,
                                 'selected_additional': 0,
                                 'shortfall': 0},
 'fk.99.com.cn': {'error_type': 'not_tested',
                  'requested_additional': 0,
                  'selected_additional': 0,
                  'shortfall': 0},
 'special.vietnamplus.vn': {'error_type': 'not_tested',
                            'requested_additional': 0,
                            'selected_additional': 0,
                            'shortfall': 0},
 'ypk.familydoctor.com.cn': {'error_type': 'not_tested',
                             'requested_additional': 0,
                             'selected_additional': 0,
                             'shortfall': 0}}

EXPECTED_URL_COUNT = 97
EXPECTED_DOMAIN_COUNT = 97
BASELINE_ARCHIVE = ''
BASELINE_ARCHIVE_SHA256 = ''
SOURCE_SHA256 = 'dee1153f5b9d9010dea18e1b7724b4915ff847b711d036bec8b933486802a971'
CRAWLER_SHA256 = '1298c51812fa9e52352b9fb39fd7086c6ddcc61874395f74e88f6d039a7cdfdb'
CRAWLER_PAYLOAD = 'c-qZ<ZIdKNdEoc{6~!vJ>R$EK^vv$c(ySU-tpsdHTFI*woMUz<YPzeZOVi!es;ZtB&CUl*U;~FP;J^gj*<lHeumcRjHpdw)cYIid{>MhJlK6Cg;hwk5%B;%j=~={`Wiwk{mHFh`lTSYHdEvr^FC@<->5DgRd4nuEoUEm@$+0(2CTS6PH&gHRMVuXblV~@d<Xvx?4(Z<_i;`I~+w;=QOJ>9P2%dur7cR6$SvvK?aI`2ESsaF5GM%Sc;YG7qT0}*X&hl1Ez03CIQI^MCXOK=N@qkLGO4rlHtcbI&H;hNoVp0r~L7@uI28%3<XGOp*={mQ^SsV>tPSeSANAX}$q)<J|!$CToPvRmT>iR`|R7{dxRZrebqgk{UXIyF+6>*VFV^v1|hT+5CH`1AOJ)TC%Bv=&5M0bM*hULQ|4OKlg@Vz&Zxq2Q)`53yPe)2!?BACWS1QjFwWgZQOFwk7Rcr8z7>UkO!WA4iwo{V9=j34AVi)O=gs(<P3<YVl*{*?~)<3j(!Mazq1pqtH)^<zPE4J)r-7=v8Qk^vwK4S{;L$R;>0ni<tSc=uYG%)0oH&nHP?70uG!w4m~ts&ETmeKwiIFVky{OY`F>%hE$wYM6n_%K#UDQN;9dQ2hw-0mTdY_<A`x@GdXPk!H~-nM8nTGMHjAR9)JOi*Pg<wpw3!?$#?eUb^Y^J^$bdzxC=he0ni`BbiL1^(#ToYkv{8N_v=kH*b3zy`Xp1gBMR+zUm!4ak=ANn*)Y^G2VSXDb}xC`m^BD6JGoI&)<IWg|0VA_G9mJ@nApgc-P0!e!RYM2>{^#d9OsHC`+V@-@5hEGcVl^uYK-0Xryf4Z?#&mi@ZfKXm_?+9{iWZFr1l2>R^^0wiR9jG7maIlBc6I1KetNr1~(~gK25wqS{adje~p~ZC-hT3k2i%NGhoMFo|)9W!s3S0^%?!#)PQ+GDzp~tnFvJeg_w2#DyGf5in_B)b)IB+jq4a73nk?guBN@oKq`Z&lm%~fVnEWpx+{PAd#8w50k9TKl1+VYymU?XkK1~=|27BDn&e<10br2qY7uyG$xP%K6#IOBR@FN+a^HNNrnKk-8l`4>D({D1czAy)Ps9WeJwv_@q7{uV(BZ7kc%TtknyFvugUNngU52ga50_dZ72t0ndL~XqkNDg{m({|9L9%aeOB~0J3&0do!s^p#c1tGzoVL*q|vZ!(Z&aA01M=8V`XvNA(GiPfrKgOSAk;1iS!b-XWl#p-w%*>w2iT3u90+L+Wk8--veCs@U&vMO`{s8fHy#Z?D4b}fuc_V7NQjekUn7CBt48XjcqgBG?V0wxQ$~KK==r_VUWe3Ea%_+>~+7(Wdx*JWdBYEo$t~Rthef&2|bI0e6ick{Ovo1cW2S-MZKNJv7jf=JLaHt={vKNOWjijs2k(iJqq$7OXlqkP2UktAHoTEGN=Qcjw&PLpF=NUtF+;OSL>H93bn)VZ2-TyE6K=f<HvQ+hqE^s!LAK(MPaP8!E!~esNYcnXU7+!$s&FZh<DcZUs@DE+P!FksCA5_!HYZv=qikGfrkxE>R_Cv@FVhO@u9c228{+Xd3n0X2C>yYekFupgnRn+;AK83@bqb$-(d*zQIHSDzz2kUs330S-Y5Frwx0}fJ^^d|9qa|LK|mjZ;&>kCKw!fJ*bZPg+EoanB%Tc0Si3`om(Id-M{{7EVmOi*nwJcL;~ILnEWo2d<H%kurn6iPIT_|WDBhCMt{ziY741{scRKu!8Yxr^W{b%LI4WSya76P#K+iymfl-(ZiZ+z#1i*C7j{$995+*>;@!hf^yqM&<@RP#@;MM?+CABa&p4I)Pmayx>8@BoND9L~pH++1|d-c`}UbHC2DG<qHFUh@pF&ZU<1n3)~^*l@AFoEW;dIzu}@ksPWFvw$XH-_)<BQ6GGq}bdVRN0N-4ekb5IH0^)NH2gmm9s}{ZVVc$;Zi^)Djaapb`MrKg~^&Ecw&|1WyTR>5||rc#b&Vc0YVoSwxh5KcJUO!!9M=dxG3iNr?=MEx4*Pb8^0uE=6|*6ANe<CG;%fNFk@TZ3Do{Xb}Cb^{sSHocz+~i*+h|nb!h5RXM1x;Q$n)_ugf-9-0>dodmGh3=8I|DT*Tl2I|rM;<2?<$Qm<DZgju`}8({KA2J-5Cis08E!vvOPpqXX(spls`LBhi-3MU`)o_l=}P1f*G07^7T2PEh)Uz5eSEwW^HLGKU;uw2exM?i}3G-AAZeE~8OOf1Sd0kWC<@o^5@GJ~xxz3+6r9=CcpPOu*^E~;itm>!jle6n)z|5T-hY4lc4<<r%5BZ{(Wd+I+Mre2;5wt8PAlHeBoBZN1Ohw6|52LB+1RmLePCBQcB6CvDjJgd`!WxKOeosj`BU|qE>K`v`A*_vH@-)>kRE8Fb+y95W*yhr9B(b560CPfvsN==e!f(NZN>(ymOa*W_`<)}h^)$nr=&4Cdgwoj1$Z|SKbvbN<B8S=RW`ic*y_J}}J;J^q{7vNBUoFW3i|MuYUF-G61*BME*#vZF)Z&SG)fPbETGdA(o?t8N1+8G}XIP=Ig>Z#YbU$T@mpIr|&Y=06>cZZReZF!lTpbWi;vH~f0iXsAV{!vk6g!9m!WEj$SK#-dtegPy`fpo6EYu1Ts;fY4C@vv=E3+l}>lJ*KP4ao@3xD+VS+FCMPV`#X~<tIRD$47(7B2NzDj%FT4o^BUUAZ)2@D32%1j5P5VaMZ$<gcGtY%+aU%ewQJ_z}3sAeV(k&j<wCoW~!@oa`9qjE-f+gnL36~%e7J+T)tk$X5pr2b{i<2>p*>$xUmdyh1k*)1JWrGeH0;uZ(CcVGw4KvKMjmiSDgzdJhm;bS0U$2s88(*!D5b#Oj}V$7@f?`jxiV9p^jqGSCw>GOcTbPfe_n1!Bu0Jd`gvKsVYoZ5Hicfm7vav?z@ju1Iq`LTdBZ+;hcs%W4IBfw7kG}!h^+Y%ID}vK1WCP=dumWVghNv#(Gedj%9epb<&3|y%i<{zVU+rh?x*%P(r(;$(?l76jjR5GBL2zrs=YUB5R@su(tqt!l*#HhKe0Tf0(a4#y9@88=qTbi`$^5Jz7^fUffxr>Kr9|SYnW7^F@yD=!C*oq}cV15%PM$C*Wda3&W<))5$>$hhA4NsAyld3>}XMNhUCy2H>NyC0Hu8L3}4kq4UG@V`Q#)<}?X5JE&+UKodemoh53k=w_&i7=26U*l*x8;Fsn)=oM22!2&_#YX6;Mb!`P>c9tQvi2yJU70OQ{Y?v)(iuRY32juL7Bwgf|<f_Ye480k=LVCQtV{t#IFx51`CTQt|=(lQM9s2>;0-FV%M;A_@Dih078tVm@&;l?VGJGi%%s^UP#5NtJMxf<Vw**i+Rp?xdJAhPK_Xb&d^^rwx^_70DAPM=mNM&&2*>w~er%6t-Rp2-D0abXQtBejqR7#=75)RMJUe9~FZ<Rp74AHcp?bbuq>yk^T0Z8_|tI_Btc4~EYQ_y_$$*K?xV^nsdnV=0d`1EC^S;xs$iJlZe(D`OICs@AYX11DU4o84HqH*q+J03ufL64M018%*mn<h&Ps1s1@;qR31N68H60H5addE>d3BM-ozRRoC&C<CZyz3gZb?S&}1K_y@!sAjzs&=^=;7zl6v6_$!?oKvf82?=DY0SRP&jI;+Axhqi58ZlFW5ar4~(sG4l7BvUWU6u|2edt{2)2UJiTtB7^>VVrG^<Qe`T>ZkHDEY$N%w<_u6G+%<=b@&si<ul!&Vl(dd_M2_htS+PD9H*_jeOE{5Hv9f&_ih(^_r*fY6PdvSk|!B8cd=*_pYOPmG{bXM}v5d3IQREs5K4Y085H61W{-*B7PW%1=>CfurBsNC2X<Spf02W$N<wr`EfBv*)q`Z7y^<;E?M^Iv#8dJv&KHaXs`p5=(xWT0FMOc64)OAS1)zFi>RLm(IF4@TRfh+bz&jKU|-K*QbV*c0pR||6$B|CSxk^GXxeIq1Ip{{INV(f0q6Tq^$Z{*7$z$Ji(*1BsxwBzp(h%XJge@jQxum_MqQ5B22>e1)YOl7GE}~bEXM3i<)bNTAi-<9;CB>Pl(5Ft*#}F1<tEV8D$@lhtaec)K2P(c0PaXBkWHFxw#KX@3D*^1i2sgmb)9EN`JeAb4&0*7539m6@u>u6zFi$siEbvO^6jdS4D>7k881&Q<_tvBu1AUnky$H?fO@TI?$8$-x?-vhS}Riah`PA`xebes6f2F2jxL5C1z?{TAw7_i^o@H2+)ZyKwekrJ<5VLWLn9tZVO$fY$tnS=wHq+OwWw}b)?!&-jM+zxl&g5Xq}|5eA|j5pwCcV99u)0LPe3WWpG(^_;Co<$*gY`|@GIP9r(NTJ&ar_CA<2%ZY^}v;V)cvOppi~ddu%fVToY=a#e1l#J|?w@fmfudF2`D-GRl<5bwQFUL|5A;46aU#(R>bU1{{IZy3w3S55Pl`7*r011T?DRSFpY;6G#bCm9h^<@QSq4Iin8v4MamwOeu!x#D5jU&9!TLz@EW|ef1i2;MxEWZW5GX71zhbbkYSrhLi~aVtqawU46aZ3!ds;T)#+9SNzjX)q|nlHqXju$lYcDBH}n5>UmNoskUv*VHYFTFqfuja~in>+ZAIuKs=U3Q06CNk>4q;l2qC>ZObd*m9omKIj|3hX$={O02q9ZpRn14*mhUn$b&}pjQ9udNjT6iU%u2i4`>(-R7SEk#NCKcjpiOB-<o5uNR@dDDNuZXj%|v`2VXaQ!InOc)@B_T!vi!00+%`mTX-R68bzUz026H{k13S%&_gSkjnZ~yF1yxnp=uijSYM%jUo8sr3QzVr4dy`Q3l=2vP#)XZo$bpz+r1t1+zIz4>25S>BGTqWE@pW&iWMBcTg3!p%gUY#Q#?MHYLG&ml=6yI<Ym0bUUJCK@rxnsF3|nN{fZWYHiL!Z<tIEX`pUzKO`k}J!yIIvSur_g4%-u>nq3*G8*;~r{Ax<4i7`O8a8mV9acuVFVXYt<yuJW@F=$CVkwvk-tD&O~vR97XTcb${rYY1&!4(7u;+cpJt1<`V245$Eg6MvWSHk0C3dF{71cxlpgTn~URj{rZ8e;63u^oU++f8B>m?`P2c}9`)0_TZz7LLwWvD2JFt;fUNMs0kmzicK}!PWE4#qP11f&s8V;CeT!=pv{K`6Q0#ZR}v>V|AXi83&KmO6MIyE^n2Rkx@^$|DWuFmdLG<am-+Y1_u33$qCn*R_dX%0%@&O9XAX2G0nU~<<l@Hpoov#sK^&)wCv!zx5)-J%>-AfJM<EoQ&GG{hcK83huV@ADCA5LM^n+ip#A<(tA29jLIwsLH39BP8;ln-Bq{}{^e3chg)QSOIU-*qC@n@s0?JluQGK<axN_yv6P@PO;YJ>3&l-Z&X)KF^B^CeFGSg}Ex~Fg|qD2agawBjj1#g&Q9L>O<h0spDQsYP)4pfc)^3GH?j>1^z-{mW`HyUY3t*Sv!I_Di%9iN-K&5NrPR!b&)83kO{`#;%6^o0A?h}JMlrEL-xxT}=P8b7%}h`x*6lC_4dvlx*7eQ+JjPB)<BjODs#6H=T?Z2^lO6v}Or=hWRBMXlvfGF3M8528u0aRo#%h8*SP&8NEFmENT#*um7r7^#v1NN;}&Wx=&;BiNld&AlZ^;yf;vj7t3!(OYEfX81Ab3=Q#md245OXK9z5@e9($0H0x!r}MVv3Fz~Z&Km=0mm}zUw?XG6k1qkCftfg@pPjS&jv|N~mkkDH8=03K$9WA-!D_S(M-@wIu6@z6-O#`+F>!GUu5M^_b}?98S1*@rfO5K>ogjuZ15GyUOk3;+g$b(<l%6+B_L7<DEP`KAvO?nqeMBefb*>Z?M}^<A7$TBG8(AVv68pOJ<BbZMa!e?Rfxey`;@HXGKjF@tZk=eX3umANJN1=2cc-(xwedu0c>z-ZxUrLW?}^W~L3-y0?G&Vk3vZ~|s#RU}1x>#4=+^rAiZnt=WC46pgtF0JJPl~L=n@0JiN7dSH}e!!39!mBZUX2ti%v>|2o!UMOsuI-U`SAuAXx&cv`u^lIazIV-MZt)y^Xq{dA{>X&9hRT&;a`WEmmbwKkHngj5FK^r>(wkVnMi!l{80<TwC2D3a8$F%hkQ@%7W~ay8BjB6ZXVfgPo`IwqK6Ha?dif4>Ep2esEC7$f;Ox5EwI2_kq2+i<Vz>aljMX)=vNr0`WhA?N`eoa7uHD2z<5x!yc&f5-FM(O=zEBpU5VW)=Hd4=LvoyoaTOlqsE+^ujW)x?V;0#Ce*6vZL_1eENb3;pzOwb<XU;A40h%@*u?hmE-{rIdh{E;&CA|J`loy58Fl7=N+@xtz736uW^x-FlmVD(&og*4#gaAi8@K>!I~)EBeKl#Op}uc%P1pI$8lLXRHu;?ur%Lm-0<M3uzvUgUJp`;Fh_tF9F@GSX+E@Ox@cmJjq^<l8)(Yvg^5gau#a@dP8;DyJo;s&14hykL+$s8|$3<re+kdF+kwW~};T0%r*VSRj<fA<H&@ab7QK!>s7@f%GJPorG*p8>c-Wz^ndscjvGM1uG1Ns+`XNmSJVQ-BoVrjqEIRk%8ZwNa6GxAG8k328{9W}9x!YXPrIm+MgZfiAU<+BIdf;zwtT<P_8x&jk`g=GBhlerZOfTbMOoMG5mE`_<J`6l+SG7Z4B&C&lL2XT1@J_tKRwHH9>90D;BIP?`?N;TDRE89~0$@DeNvUaL;sN!bRIxijz7~>T=>&R;NYpcJCxOS-r9LlBMCjPmEfBp>rJjq^F7KM31ZO$cvLU}w&m~lzO1=e;6XJ?7@@A6b@W;L2U-y7t4o3}$@UNKE_^s`q#;%_{3GDI+o;&UdG+2CDfL<-`%k+)bG0!eg^!VCSz#qys<lb5f;0D;#A*H<0SK(UKYe*%1}2&)=OE!!)KA24_-+$q}FL#>}jSyG!^jHvd|1P&<RrOI1_WSUMxQ~|Q&(1}Xyd0R7G%j$8wd%KL8#B)4vq<eY-umTMo9~}5q(R1`JgVAiG*V+2h4Po1{Me6;44Scy#-_p3gBX*EHVNj?3`L+(gOhi8^y!7X9-+tNC&^qVYp9bS^4Gc8)piLqazf#$2PxN~KFNa$q^XlVe6^l7ImMrVCO@n;n6m8Be3pEI5tDg2sDu$2B{mpiGfe&3{$~xjg-XuP|y=DkVTl3MPTGhP+P1P$5y>m7SwJR<+1Y3jFED1O%yz&AAJ^LWW!?4{5v)bLxiX1VLZqrLyS>;_a+V<MU&Y2Xfg7bBD%S}y|ttOf6%e6;)d0iXcKziyspuV4+%7o<yQ-ji?bp`~@4VN6M%g|qT*HGv5`%W#+)(ShVVo=xo&@!XC6tb6lH-cm&BMgk+^e$fX+Ny7Bph*ICk_2`_vI9&Yf>z~V^Gc`5=q$kgNYnEpS(r`i#)Sm^D6CH`i+f!(#OrFf%!B2Il`^X<xcqbJqwHO!`hkOS$O`znCKogsv8#fD#-_f00<p*zO!|#NiKuR|`DSTh5P_a594CbsPOF|zTE8h>6Ahr@VL}`U7b`tXOt+^-k0()7<MDbj7oVC_h=qqAv$z0p-Z*HLm&6h!QfW-Ky+>?^y_)J=Y2HIYaK}ta<E`4n4NG;q?epn2AU^Hz1f*@F)Z6UDIU99cw1r!2uv;+9Z3eVcy3v%D0~vVB%`?_k*z3ChyG~**&K_FU<|K`ILo`Z-u&D`XplfsO+vFCJ$Qx7}Ug{p8e+;E+cH%T<+hGSX9Z%hqM0Jq&Ae4708?f`2&chHpPz*UeHN$Qm&xqbIlu2^nVvtC?Ns9$$CdvO&qEM&x@gSIC3!+2<C4`b}@vi8WTew0P^?RJgMvqdaVn%!Fu~kL%Gn~JKZPApRlpl&AkC_Xtdd`Z}Hi%{+sRP?aN-$SOkY>3m$;lmtWGjo*vJtYco5H#wy)+x6z1A}#&}olK9i?5zw^k){I8+5m0Y0cI6;4O>1^27iNU5}}8!0i3%C#d&>gW~HM-fk+;pK?YYNn%!rC=)Gjydr|CD<}7EiO-17QktuP(x9^;3aaa)6vP8i7ZvFW;4%Yq#>}JtZn4Q7^_nUnXh>tJG-M$-NIin!*-C9#k1OUg5?97->6f&L6L>4v8`yzj%$-Ot8eR-`LR_sl{MN>ml#jaZ#`RPC{tZs)k4;2@OmU;mzcVg#f#Ch(^{hGHigKYdU#sZ8)OAJd#Uzc>DL`n)upObWng4gm}-`yn#m;KC-}hchl(EC@*FrXQgYCvL95B~CtN#;P||@Who}Y>bXEPvS}OE;>fxVF@_xcUmsxsN!+v^|Q9tIrkOGg))-LTX1I#G+04hNVo$4b<4-1wtiuYur^<oyR!lpgC{ArH=9IO~P89s47uUB+|BK<pvNLn}~hdb>Urlmqkn#*#iNP?X|;nLTIN}?3YEVd5QQ0eJP=hnw&)&1iwxek*ooK|~envL>e4N?&&Bi8$tw%X2VP1gC??MpiB`_arwG|y#nB_tBo`B0?rliNU3GZWG+8HVE|F1Zk{v}&dWVty=lHkYrdcy-Vg@O2v4bSh~)CvarXREkT;|NJ~4wF<H!OB2rQE`?(3BL57TYGclQh9UvD5ra-48BF3eNGvvufOL%GMMepSDVwxl5=4`g3>876=s2Ikn1;y#p+G`0O&j!glm1?!znAIn75e)G{w>6YKTy_vf{4jjV*j*c{|-PCS%Cg_XtI2dd6T0lW>cKvxO0e7JHYr^V4NnTrUL}M>b-=?1!J7|VS;T$Mg^pu$fV1-IhkgwIo%(&9{^O&`R9tgE8{w%JvshVrA{r;82WWVG1^`E0FrhVG#AD3gf>_*%0MW_r&(%zKA>HPsyu&r7fSH5O0eCjTcAwa&HQRzTv)Xe%3P*;B5)8eJ<Dq8LY<cFU!P!t<{1I8>+`Zwff}ixw*f}WJ7Q6+cFU6Pxh;q+VMdUpWA^TPey4Hv&YrGTIU=<c%>=JTf?4spW*|D*E$B7Jpves9i(M7MNZ!6}2#JWxU408}rRcM!XpLa6pL6mAA$(m(t%{riLdco>0rykPQG_{naOdkyEdX#EV^hJ=oLpVj*Dr2=CK>iGD9R6BBE^NBGVd4$JgB;quw<i_9vU?gE!!tb)qvPI)l#6FD!?WiuhBTFEh!(VGa3?;y6?#(t78CK!>|sblp?0{)KH>cIE;rtHXv>1&At0?9eeM6=e@7a_U_~tx8Q&6JHwNW?&j&b@$co+PW#iJxHIfnxqPLo1o+I3nxa89|2GhhIk|BYsBLta@jp3S|KxQ2lYaq8XbCOsoZx48rJJqW=qBqC;^|XALvF=WU?V*WbI+y}!$3%c`RL$;I82bW$|V7&;V=n_pGil9&qP)gvPsJS`LRyq52qCVHc^p+sQ9(LmqrvHJERCC%0VX;Xl;jaK7j2Q%`nMPJ&1WlLS8$e$wMh-vA<|@UE)jV1IO%Ql4zE^JEl&9_Zwg%Zk;vG{Sk(c!6aV8glBs&g=@0A$U#Gzs#9Je+iI)RN$%4UnDr`(Z@MM|S|<>CtMP_dA79?amVy1jG`TbAPWys)p<9xvwS11rtv^GsK>-L~=`VqW=bZ{e4Q8JG#@3=oY>emuHn#m4h+Q95+bm}o(<-#73nE+1j@z0BP?yc^uX1Dqe@^*XXmW4gxaqxc<Nh~qdVl`vwVVDqoEH%*d0G%HY1(bg8F$lO9-60N2KmsC=t3V9KeG>*pA&_rR55LnJAtN92}rvN3h?&p1m-y-#Z{93bU>+gIP_9&b~;i_^`kmh%6%>+*Z=%KT=Qh@NKawxap*yknGEKNn_L7>J4zUZ1AI>>_Qqx}%J<RD5l`;J!^6OMW9(N_QqV0W4M!JbAk0Bjy0GKdG_$&{y)4T>J{k`1LIV7KtXkPL(JVjYBk^NvXO?5t(9xKjOnP+H)5&#0nbUz3SqT~%2!^*;hC?NAOxwCRaqSl6mXYr?))iFG80U2jDS)gQJnS^B=-Hzpva$qeV8K4WVIhq`IbmP0Et8w0H2q9ywpAA(<lgKuIB3+xbDw&_d2e~9xLY`@YQ6Xk(WedZpYa`#BzYSsz91O}Z<X-%2seFrKth=0<x<@XLGH~|>MZrr)kUc%V^MK4#P}J*TT+u;QdMi1u$o4N3{&I!cA5u>UFURQDFNK}ACnypmW4oXG#qhYZwE+@SSK@s**>U|O)_J=F~|%IB56h!U^#9>|1LPT8&_O)uQ`|>24L;8M`D4_&jvLrD;}90QfZPE7lRo`v()5YJQ*6p!|N94V18|X-r#475y}>WG-uYxrab}2`k<nt-S$8HZ~yl2YybMeH-GYnuYC9a{Pw@%%FC<o|MiDIdkbH6Ixcsz>Vdr?H5qTvcMCm4NYUMt8L6F7E{VI2H&xn%|8)Ok<b}7078`0b#@g|Ex1VcI%Z_q+!mC!2DO?Rk#g$Q~M4UVuP`cM>QdsZtcSG<~ouAWZm9e`3C#QRUM@UIsq_T5pn#x;~*Mhkxhi?Nbcl}Sk{Oo7fWb;&V81W7<fYp7s^)*^kP&kiluNisCH#tdq1@0gwtt8E|kFAhp2OeyEo8vU&kz5{l_=;{zBBa8M-)tmcijH3?3Hxxg<HFn%jhEOdoq(4Xh`$I0>3Yl0TIyM+LG5j$tV{ud#9G!l#$ZsN4O;)>a4u-2j;c9lhvlKsxTw57Oni317ynGxDzVm8`RYqST!8JZP0L?j$|;Y1@2m=2t)*#%K(wJ`?4cE0Vxy(HGvnn9ttli@Kp4U#LR%7T;TZ1i(bvv_>J&s+R@GugSxjdD`d0X7;(hL!%}X1by!wDg*N-!}=?oOdEzz<gD7h0UjxxF+Mc#sfr4RqxzH{M@zw_z#_MOAEoyR+Vb^k!k8(LBzmnp8kW!m~5p%tFkjVCTWd3i_2V9<?Kq=Vf9<cs}|*}l3i3mOvO!5-*3=6DB%wwR1`cZjTNdX4sXRiAi}y<MC*Y-?|KAOAU8!v|w^jrO3-#*@8EJw6Q)LEsRbFH{VPsLOE-;1-ifI-1h0+-Q8OIRjA*teO^8mz2~QIPDlGnzA6j%2#SVOEn~!zzb=*znHsH3Do+koqn~U!MN=&Q1rFzA<@-7IfSHi`&7sZvA!=UY3b%SMP)dx%-IJ3B`+T$onS*jc?%oArz;dxl3<7eQVHx$KTFQ4Y+Su2+FsTN(BKF&Bvl~yo(w69K9*z;U|9sR<D!jaA^cJ$j5a|uCY6XHYMgT&Xs;omnvqXZy06LnByXy%DC!X^T!A_U+Su!j@T?b_s%q813hIEaw=y43Y6;S@s0%*<kqpLqAgS3KlqOi3t^(q#ybLt#R9vzWt$1;=;PTB{Q&(^}mD-jPJDl4ZTGMarEb;VBS)8S^nHsI?RH%T-DEV$r@T{Cn(8HSA)7#Nn6EYyN_Bhx}4}yc4zv>(#hm5M9=9z3&yJsOMc51nuq%O2=gI*4gZ&uF{kIu{-8Y^i(YTGtgSYd~dp@X<CP(Ck{Z43<riZbVI3`izu`On#qEm-zzf@LX0Iu-m5ffE0d`;nRj5<N&fg$yptus(N;f%AM+T;o~j;d@d>Du*&agj)F0WQgg5L=17E$E8`y!cqw>w^1M4=~u3O?0TC1^%d^2=(6cAI}mCzO&@0;I{+Y+D1!YnzH;A}L3K$!a7P`WNKi!%wy6g$1l?Jo<18liwUDIH%odT<TNaRvWpoKY9~Vgxd4Q`ZZM7@a|CU98tQ#drvtqqAFD4-MfOd&=l?dpzSvq5Irq|`ph*}-9v`*bBHB3HSipzZlAr8mz5iC78Fe!d~=vRyccIDD*4-G$*>K1e0JFe(()xlsWYC~q$ykeoe1GOs#7Toy*73bRZD*Ohy7q!$?40-AOfh0pq_{{0QMrl4lc=GQoE?w^Rd<@PU;2(JL)Mzx~t6v8y99fyjDb#r?t}|##AQ=szK%F7wT0oy-a2b*?2M}Eg)}GUKf&scm&&QG616IPXSHzfK97As1rkdNgsY+z9zI95KrOZ2<;>rl!hFgCS*arBwQMnke#G9dDS~iRy)Jud%9U#)y6Zz{Ea_{iQnAg_BJKwK@ESBUdZ5{cvB)b{ex+cKbEKtm%qXNCIvOw(+oPJ}x;9(bRmKdXJ!eYEJ4zV>BBq-Le6q@|)_r7}nNAG>-{!j1!>ixIh|K)psdH*lp`}X~B-v8eFZ@vG{j<R^W0Usn5V3R_a#en$)0s+x`Rvakgo=m226%P?poYu`A&6uue^qgn#l7#!BV$ofU^?`!WB^S(WOifqbss?xJs%{lJ`8E<toLSRSqN{ov&Z346O%=S{vpk>BD2Ml8;Ki(CWV6stC+GscyO1jx{Wr-W6i;JaSW$Ehsy5~2gw8<qVSP5ZJR}5~u2By{mOD%loybR1r72%B8w&s1(qynC1%ST2-bNDuksb0yju*F;5K8gImNw`-5Yp7sq-iX)sdrh^rm>TFU~d}xqGTMzX!g~zX*_IkU<$|=0u}-TDZs&QAL&=ROD?_SAcgvue44`j@_fljm+v^kM8e{v2Oi%s#mbphSKc}T55k=N4}SfPhd=w)!{7Yw;dj3I@E1S+{cqnw8vgKWzyAHb?|<;kzxn8$Z+!Uezxwcd|4eVcb??LP{r$tg{l^dA{oTW_z4_6ve)I6Z{NqRO{0P5&@Pq&S!4Llaqc`7q_}#CNp8S7(=fj`<@ZmeZ{pjmIt+P%3?|<{lhi`uW;Sc}u;V*tbp#S@u55NBF5C7!{AN}fA5C7_G4}b9+eEZ(F(Y3~#jUT-G^Eo)$Jh=B&Z}<K`%^=bD0$!05;PvtSpTmp%{@bt^yAR&|aq3Oso3Y5ajyz<wShRNKE13`4JC~Z4Z8rrKXhsWC$0K#ZJnaRSuQWlx1hrfcgJ+V|LS#A|Y#MCW<}x7K(3#*sVN)rIl%NR^Xg9+d`aOr~AhS`=+7WKPtPyNe`FcOkqN_yakR~jpij5rsK(Ym-!Wn%a2F>F0$wX1%{5vZVU%1TWN^d#oGes<$tE&PRyG2FZw_#ElBBvn+WB|+PURG1h`C>8vfh!#qH}D!i2!?=b$S0t>%<fL}bd7vU>8aqyr$2FL8`?@ncb}#gpSt@ri0q$|qg}L}O=C{Ur4~sYZnDK3HbA71f<yA>=tZB=#Si|M+MnKnlqh#06Ukj<W8tbJ&Jl9=HN1%gZir8j*x$t}Gg2+fpoe$=@b>@s!{7br_y6|h!=L{ED3tvE;BSBW!8iW*!>|AHZh=ui-v0RhuaL4nxc6P41P|VQJGrY~-T$X45Y-3o{^Z!BvK_xZcUF&uAy(X;Oa$8*0TEFD&LMEd)Pt8L-rRbeF36#Hynt)5?ox(8B-HR4u0Y1S1z|1X8964u79B)<N)Rvz=GX_J{O%uq`mK-N`O1fHedX>TnnSo-WG_eKdhiI~|DnnRj{QA#oa$StO8G&npoo)~j2t2<R1A-#gK)sc%|+F>T6Q!OOC7KAWnC@H=2H<+wsibcMONBk$f(nU4X6z@vx=HgaeB<+ZD|Iv%$kHKB@}tf<SEcO-6Yprip*D9uqd+$2x+>gU(2f}C4iPrV+o&?JWd@j6|n@yahYHTfJ8r#$Bk2)W~8^4mDLw6;_bOu#+NXEJO61ltErw-AJG9h9oC~R&CF<N{U!EHx)gA%WqDI${qi`>Bu4iN_N_|`t?wi(>$G{2FHcnjQ`Qbf9&11tY79B?40j?k37*b>M^INyf>*h6Pv4^0SFIW;($s^I4UfN42l}FOJFW-#TK7v<1MVEDzv{L+s$zz%Ibj(se{2^vF1e_2l?xgl^KPktB^ET`?rLDZ(7$GV3`CubfykA3L?T63OF@FlhYLlP3Gl^1Yqi@R)o@HjhfIW`IklkMTEwC~HVMlrQ`tx?x6WB;`BpdQMRGy~L#(=2I<z9i`w(ke&%$%M<(<HbQ(v;Qby{`EcyE`e!}~%KR!$qDa~U?>aJ>9tS(^~(+otf9N#yA6TcK%uvqX6#uD&Wl-5qJ)FYCHzS@Sz*T=0v!Kvi?Qay#(O(!1CB_Hv>_aO}VtH&i#FN$`C-S1zG=huT)<T41_cOD@{#l=sES8$2$p8?~!f_1S5q>zD8G-HZ*F>DkvaHr$iP7c;uAzC!?~D(sIB4mk(Vbj(j}%79Djr><heXML}rhv$_!QL<!1WatXI5Y4BXD^kq&{e|aly>jEFn|N}ur0lrCISlDJq+7JHNp(3iw_kS)GUl_=Hy>}z3ZwGku$KEyKwZSj&~WjhIu)%8{~SQ(#~`1~!g8GXaURGoWyj$_4wN@3ev8YfpDvgnZHYt5$$U#JYM==${br;A^i$C;D(#X*z}Ai<K=VTmwzJK?;uh06<#m@6TQ(Q-D{{XfZN!$oE)32|b(u7Au;mWMZ?qHD;xF5HM57@(KyI>>y?52atBWHzISw&++2f-CqfOwD5%+V9HG#O48M-m5y=qW1<`PsF)K)$uwXb24r!>y&y-$~fz|=V@3t!#LW-d|m^c8<+sf$@HENob*CpYf*#$K2%D<CD%b!O$fuXwjNd6=l$4Oe^T(Lroq;9XX!hvs78hHJd#+auoMeZ3r|CJ9R}^QOj}cX=bfB`YL7GE&+ux!!zfmoU4t=pxO|)QwxRtoo|t^iU9s^Q+yWf8drAK#4I}Ex=(wYUl%uC=)0YK~-~TB+=67B2U{}W<f?F6_VTcu9Heo#gUSW-)*|r@nr~sdG074EMTv+u`=dEz-VuRCO_EA4@w%s<bo2DBf8>;AYbfGF)CBZE137BDzKO+ExoogtUx?neAE@#l)7-Xn8uV}CPE_0>gZux8V5RGP27Rd`vM?fkD2PNkvu2cgpG<L+`+4uHfr~<Vt2exdYe~JJ=OD65Ahed`N>MvamR5XweEe0MjEjH9;gWOEj4Z*6Q~Rq6Mbv7YCy|t%F7T(P`#vDY)e4vk;5SETW}K?L=ODG6!n^y!r;-*5YvBD0P_QfMMjJm5KhX9)-J_M2+n!w6~2iNqVo<Q_k50GIB-7n4i8+;ZI)mPJ-Vp`-1`8VrV%;+!KICqblsC%$$^2-z&t=B*O5P_)JKQ{6pd37QRX(kL|QsU$SKv29^Ym6Q#b&Iro{k_zVr&hiXl3abc+>vO1=~E2S3{Cr9kqTX4gO;d7a1ce!FMMbmoans!RyBy0lrbyNYYeG(Q+ru6J`LeBp950p#r>eZmwQDe~{~{e|blo6o#({dwFAt~^4Lf~7G73zNB9KpucS6he1Rgt7r2wI1aRA=@Hg$W;&}iKz0WUy;WG1o@{;@m#HEO{tSbnTye2R;+U41r~$lK-A=U=*qhnUb_B#_}qVlC~V{R&1e4I&O~i^$4NXHF24n+0qxF%dgsHZS8qD;$(dm8jHP+?rV3QY66jZ4mGx*_Yr>;Rl)>(lH3U>i<vw<S5vUZ-h^x5h;-!RHiTS<?n|Q>>-O<O!ApZZ7We;<kC)JncQYAaBr;{6{Yi*m@(Rl+0mRU<Muc3WbGMy!SJfSm#<K!<Zvz)hBm3rgZb-2lA8MB#H8O<%{J)o@QB(-CC$6{OQC2c;GaDnxf5JyDpG8d??-|`FFFI-QvY%wPm6$C(;r_&3yL9pWvi5G0q;_hb3qo<~mmQI;U?3Jb!qjDIxnx+(yOmoQmN!cC{_cCT2l_X@WK*d7PvE80_T87_TGbmFrXlVwyKDp`V6kkWG8A?2fF?3~e0fMLNS)52_x>wt?#elb~Q>s0`Wa(r+CP~BrblWEtJHi;%9q9n>j3Mrj;y2pxL!9zRrX@!uZqUruM4pF3w0C<OJ7*3DI;6eUL)9WBY~wZsJ`T7#s4h?tc4V?9gl$PeT!@M($_C&U<VG*Vty^}eQ!fqscD?`<l9ktW)C)%yKpl|5z%3%yrO$(CpRf?k0bkElIRaTjF1&jjq7R}?TiJ9D@$%_>F=53wT~AKaD<==A_@N$KxzJBgmeceo%$rf2TVPb@=TZbiyN_-}jt{G??P?7fM41aqz;Ef1_~lSAdQ1PH{7n<K`k|@LD+MGGezJUtJFKxwIqa%=7aP;C6k_Qu$pBZOFss5gq&FfTN*p(PZGuus&<^8dZ_K0%`*gHzPUjA!bc8o076RJAZ#}I$B?F?64RF9Jfs&S9al{g#d1$jO#(BB5p`(Uvb;m&_Iu(t>iMhVy<wywCaji5ojPI2t(kd@Ll3b?dJw(>7&7&h(G-X%gwKGj&;*<fFHG5k)V4A3fO_xLLT3A8>?TWzE!H9wUY;2N(XZA~5$yWg>Mb8APZw00uz7l#OAP1zXP2Peust_amHKJekD7Z487LGP76MntB49m9jqb^^(MLRCEbs_+K3d^I8&o#B;Ld5IU6fRIg<gkMM8c|@!nY*C|MQ!=e0tZkuEs%4Y)<LPF)MlY9HlXxaP6s+=d||x<bZ(O;6NJDGlV~Q67ZlA23PyU@77C%b2%;9N0i8c}h35q8Y*T{!L7Yt5!Bba&ppo0Nz{b75v9n?eAou_dg?iU1YI3#IMhQEB>VH&Mp)nxIDZ3xJdGXvpRe3sXFn20kWid36m_ea@%5(Ep4B&^0L4vM3+$r$6jUn{Z!|TKW#Yp6WL_!pb;LSnT<cX0jzba|iFc&Me&;CJ@B}gUiA(&d`vc@kgN6Wg1gg)37m<BX5Z4D`0I%(&^&PeTCQ(#+~Tt5EKWL~OSv;bJ560AJ}1EWkF8jM3c!LDPF-#AS$PqHrRO7n?h*Srm6KV<^5FXmddnkAGc5M@j>mMV#^^sq6!l@y6SwdYC+)ZAW0PX^zV4Ih|XiUUCpi9XwZ)kz9!f;{ZuP2Zd93#ejeq(87fH1a68uR&iMg2sI}#%(G#v!MWF{E0L92`B{~F1{hiG55>Gx-ytb@Pz-E_xyu*e}_qTz&)2WJCbwv3&}G{`r?gS6gL7Yqzh}__3?vyZ_Yr5H8ArOFWJ2cTju^-<hJ*dLcS9kpI-F__x}ZF^1;2ogAYpAuQ?0QKwPLy0k%Y!Zvhu!R?(cr4nQrk`#%S#;-iJpVtIsMfd}{gncR|pI75>p6q~cNvg*I4|Lz`KfAH=<%)BE&gbDg0njH~}hTWI}{KOkBppD%J_x=+oNa}^<;BOe|?FaY%Cp2;IukB6(0YQ}qL~s0<CXc3F18q3icD91ePr94AIllj|Q=GNmNG3lu3vzhESPmY%`!f(plLz<S;f3@Dh!Z!S_4Xdz`_7;S)+eYxo|T$`$^Ey=WV5jH<M)3Vfo>MA5<u_G-+B80`>WUseEZ<uKLBoke`nx5&wieBDD>k5#>~aY90H<@?-<U@0Q&v6sm}ek=j9Allx>V*gbXst;D7$Jh4M88^nld@mIc0GlqaCf0Q~Hrs|Yd$Lp36`h_V;D^{guXUGIzWE@dLRi!1pz02G()uJ<CfbJzQPxi7GYiM8<inwq)0xXbQ(YinyB{oPvrk6#L?CKqAE=->udgmQG@u6J@&66#Zrf18qvdRO9-dQ-{<`jw%OQLjpw2i{yTs0>E?V<)yL(C)FvU=Co?Dec_-w-X^Q{+GY}WjVDC6P-qPn<d`ql6h`c^fx<P)$ckJ-gwmBb)Y=Fx8T;#_ka1|-j8Q{t2JAR|8qC1v_W-0M<4PJzHsCAUxc@wd-<hXw}UC2OFHD1L&<77y#^KsUDHR%1lu}txJbeWe;2aIR{TpvJFd#R+;iM&_c|v>Hbrq>sn)hVR_aQkMoo}Q-{aEJIJrCUby4Pls$I916J6HR%85?eBE?^xf+`CVt$;5TB@AKft!>KHW)8NQnwAbQnO@Igx>SXHUBs1#ElupYK{q~Um^PI1V~`2HyfTEZWx^c)5)E|YS#5{qH{F^!MHUyVG|*9pl?BEK&kjw#OUJGZ!755Y1|qpEwBL@>5j!n7()Crm{>q{sm$8Ha=FK(}oQvr`j09YYGD-zfTIeZD6j<dYfw$SEZi9`<@R%PjiewUCk~jQbjtwDpMH0Ygd+HPe$-Tj4BsW#Ajb0Mz<N&X3>Xs;Hd5Y?{q2+tmcX-}45tJ$&QNQ(q8S5lhy10CEj0vE0YPCvT&%UTiDbMK+B-BcYQhlqm6KNr%ZCLMeBfhs0g;qA;DHln~bj7jbA{Ln=5TjWq*qfxgZU3Sye=Stpwv3_coM~{7v9cCk9G}UIK;|1WG~MN5?l6bJgyujy1mOc${P}Wo82U=}Jv~s>htm4hs@_nJ7mQS_W_ahkWg~03t!?CQZw__24b9=?CFV2fFSc8;R_WU##E`dqJX|{9z2_hcXxK>b+Ma!bN;GYFBJB*ZHUU;l!7>Ht*6J4yeQZGk_51_Q0D|UX?qC8}T7xBC{thGUzx(tMO~X8Yx(LVUA0>UnSsikvlW4d)YDu_7GbWrFdeQaluJlW(xE>Q?pLo=3I`vwPixBJH4djl-hp!#vu*?lldEg<Bha!?X;5cBGE<2EV*Rnn0Z{@Cc%+reJFYtT)kPNV)vbyY|>5fVT(GX%f7Q6f_$4Zdc43m;Ybfve7lQgK!oXktRRFkVgY%q)=%2Z6?oW4cUowmQW#^ll00SkT1`xUKIGWAfHZ~y=v<wYY53`THwsNNhT<-x#4Ogx!jK%QzABVHOS#dNn&y^_*a)~h6VM*&DbR<&_FnfLu0n8z&#2S{Kpv%DzTXMZWMbnbu!0YL&wL-8o9Jm0a?2r-Mnins<0oUb6=!Z#8sg|~DpC|Gv1n9#AyQKxG{20Z|czHD*@2ugywCCEV%%Jk&mUppK01p}H<gw%H*$Kr~Yv3q?Rmx?1iqz*WsMP{v_5h--chpV+qjT2Ao<nqd_80BSomfxXkFSrlFbz(-tXS5-)JZmakr(uLwnujQWkX<?h3dZrA2?ei7lc*p!l&8*yB9@Ic3?DjwSHLfQ4P-=P)|=7R^fPS6U8c~Fg1q$>8P?La_=q`!n99ChY0I3qgWLg;hB8XWB*z8)gRt|q7RKyIHNV`S8<f}pd7w%WwHc{E&r!hGISj;b9ORC~%zsN`>-u)iDF*!Sf~gDwYzrWnSLMFg8Saz42w(Z-zychx_7;V)s3=KdX~K|FQ-s3Opzmy63{hUk;~V3N07>h@TuWuUYj;}z7cZz_F#'

PROBE_KIND = 'all_hosts_public_pages'
SAMPLING_METHOD = 'equal_host_reservoir'
PRIORITY_SHA256 = '99547020075067df140b464a69540b8fef2a359329d100d336d14ff33f7b9c52'
PRIORITY_PAYLOAD = 'c-qZ9O_SS3c6V3#KOouzK$*mFM%G5AM5nZ)ajaUcq>{$=rp8k!kOWD1NPq^w`S8@7bIvihRHbsuA%~oH4>@cq|HJ<z@4bHg1vJ2+q)kQT@(_*g*WIsQzkdC`@ZiCN_v?7I@;2MaDxD5C@gni!=`<-zZ<?<+agmmJHu5G#epM!gw@l($Qj}hrdGC^+w@F!*eJ|fsX`aO^?|rgLvUIx+A3S*Q`DgPYUjvYAx-E(%tHSxV+7=14U2pQD@=lip0DGI~tCQ<wx~=k}@5N<=y{?ifnP~u3a$T*`iENl{S1T_4{4*|_RyUiZRF8{a(v5of31*-k)^W8|4|&O*Y@pQW;jLMzNYu}AxvkQbe%wwrMUJ@AMK}6)m97(pzb#e(J_K}@iR=W*e#+CVk3VL~yoeWS#;a8_MU*OB!TWf-s%GgFru_M5-=DmD|MJaiZ|wP-BF&4mx*5PoCduIPvH$sJ-<~{w0X5+D?dP9)@Xvn^E4r!tk>?}U4wlt=)sHtDSn(M9Ik?7>FRtsQ>($er$HVZe{=<WZ^y`V=w;&E)#o1yDG=rdjSq_e0_kUS(n<T3>`EIi)fO-Zmvl7<3NCuLEu<;>QyTi$S{u#{H%aZCUFD{~_DDon@NV8cmE#j+H0#uS*PkSQ)be2?cx`LUORS{r$xXQ1RBIt?N>D)`pG%Kq(n<n_a?=h6H;LbxRO^ATY@csK&C)l5aj{k}Ts5bNbvRsM5_#XuNIL|Uz)Kj2|d|SCXn2H)vC8~T_H}7Ceb@YKduGvAH-2}9y8JqLG@aQ@4vv{4rp3PTVgfHgb_xuG^oXv_fo9Esfnj|wU6~DYH_o&Hb;`dyGm}O-hlQ6J#4tqW=84Cc=7!ZudpRV!}fb)A8jCYk*OPB|Cwt=}PMvHdzLc>UaGI}?F0<fgIHzZN}gI20zmFxv_bStqZH_LPhEIi2;Nre16U?$!7Ci(0}b|5)7Fog@bNk-l!E%rqpe!!HuPMB87x(s@Eq9s*^b_nfkbT~Zv(tGIPpL2V&P4oOBO#mmWJf1=QZ{k&%DBJ`vz<>tiJ@cL%esu(7H}fX`ET3-Icsz*agn9z2@q+)baxn2%x!_Hq=tb8?8nIq)R`Jw;W>F%s*%h!i$zPHE>DN_pGqQWHiX?#<VYLt&1Gh=Dg_w?A^Ac!avck%~=ZDDP18f{#;}H}12M5^Z-~ich&^ycWDnN26;q)7g%4v~qsvga?_NKa7CB1XlPa$@_32MF8QIELhks!JckA}Lwpg}MAK3;7TVt0Kpv4jTt7SE`n!3JHuoEF=}qEShCA@GO5pUwyCv`QCoCD?<nn6ozq^eCeB?OtdhpH*qKO1}PSTUK7|t#hQ1HqxL5o1_3rUc;(*bh^Ni=v^j7I_JZzvoZG;R~2+4xWLcbv`A|7WzD~*w_gFLcTz5+5+ng=9n6zzy2OKO3z&>xM`mi9gJARmm|R={n_MQ$J58px0eQ%~T0*h6LGDsq;*`EUJ$>sjcsS|+qe8?&kWlS1&9}I-x#Lg{#M+p^4>S*ujd<7L-_1?X>w73-1nMm}U^u|*Dj1F6AtMhDfeMaYS-bCD^aMK*X9uDG23LNX6czrBx78B=Rt2nUlw_B_<S`(<T+<LiM2goqR2Ua_WRHidHO47gHHVKtdN!Iu;{Fmfvt*^PhRkCnaVKXiP~M7Kew2s`x0@L(PEg02vl5GGeedDJTmL&)>cMj)6_AEB-Q3kc$mL=X{8+2RSdx__(rT3mWvY%qU~RmJJ~br+m$b;q4V)UtLvXbvX~?6);2b5B2}A80EsZ^N6{l4MaA9`=`_;0Z+DpkLpaT{daJWjd1nvs3Th|-4vr6b0)=3_@K%CKhH4CWOnGZ6yf8MA47L~BMvTAiRM*h{Kjm?dgd0AO4c)nSF1)%X*4%azw<~&QMk^*qx@Nykr2S>1ya#-FM`t%2Wp0C4gmLi7=hJ7!5+*30PGdK#OxFVj3ljwWHo;ScgwGd04eBp(M-q&M=0#V7pG>@Dx7y(j2OaK;kj%U#^u$5VUl^smt=|w)D(<zY5O5k1VIL(m9u#n<SwzGv2Y7FLX9OqU?v~s|6^K4d9XGOjRX&sCE-Xop^;jUaI$%bC^tYbr*BbuAiIDinQc`<WPLK))?5w4xv%4D&9kL9L5PXRVo-)8n|LgfY}ZMZSzRXL`K^*zyFG)s%IFMgN4!M5KoW5A|I7i1m^?E_fa7^THDUSvQ%>9nNt8@W5c*Z^M!f=MQ@do;_QB#Sg_r6{$&W3KoGPe>vS$0k{$L<SrR1)Q@$NMlKN+iHp&T?(J$vvYia`N7=j$%Py^kfCeD0N>3NfM}i1@C6|am^Vy*N+;yJhEOj7e1e!y9YsR1K!RyRcU0fmFR#U|kG<f9aFb*+Aj@|6fVBu#AMP`Ggfd1X)E3-{WSp#gz#AfAiZ}MhvG;vAw=GN}#YD(3fhmBb{-iEq(F|eluHqXQ)oZwN+3y?37&VdE6ki?CCB2{Vg3`{FSP=C#IV>EoVu8V46C_KLlHQK-FA0uq-8XxbGK6&ogM=VP04Me=0U;QO@b<3P0yIv`8j=vnk_{_VA8L-v?@HKMhdsww0m=29Y4Y7s$3Ps&2z+@#jU)t=f!<l{7I3+YK_~(aGf80liAT)UcVYcf(DN-(@<>uRidSM@P{zNg46B$N;jRf%EsWgs=OO5UU;~Q7&@l*RH?2I(S`IjLLEYn|=WcpMNq4K|R&k*<>6AHBjyG<Fhr=P?Px+7g5ZLD*?U=E-dp;T@pH*XZW=Yx@%OC9<E@<c#n3ztadZF*Vc`vSTpi;$VEeO_219jY0hyo<zBgMm5>e|jmkB7tazNfSrpjnB_a*VoIp^S?nR3Jsseek6<edArqKNpdupnzVr5fTe5uyh+dF`Cj!W4{k1iEA0A#v&x{Qaa&@wp;lgka#S#0$9~h6ETo`swF!or+N|7l27h7bdJbTp}Zpb<+CQ2#Md7!p1bj~``+B%ny1Z;g(^-Dk&vJj6s~`0LrAl>8iFTUgfECw(vm9iNv>jKj@7STE9S+PA{|LhR`14^WYP`szD4lg_nfk}o!=*&Iuj`G^`20jabXj_RxmW^8Gvo4WB>|}<1#@bKnB;H^fGyYa;$x0Ze>8rs@g>2SQyLGK^?CQV~om)PQ~U9ZUjrDRHQ<cLFT5c8X#xG!&~YZ@i*#d64Htj{s-~A5|4HL7d#g+9O*7e2Cx;Oh7d6KmK23haG;Lko_FBQAKXd_Hifmli*6-~XXyfJ31#G6xcKEk6TGY%SNR%*@ySh<lmT^wDrkT%`|p$KEJPz5ptU5ReQ%BIaXjqpS{(^xEPsSx(A(r?dQBG*C@h!p(c|BaT|;jJUgvCL*=b$A96srLUmkr$mw2F0@0oi2%=_(d=;YfDpxq2`)Vh)XDqQhCZ0X~;zy!=d7qxqb#qC_XjJH69TVQPId5?K>obu-IQMrM~aRdVDSc+v`d+wv?rW<mdHz*|8<x16Ktgb<5kA`r4cr-l1|2@M0{Vo3QiBQ#B$ap0=gKLRN)dNU@#bWEPmRcFgnTBxQ>yp&@9<N39TOWC|bs?y7c$A&Ov$3x5Li})XBYkO>jU9EIWU7?<cm#A=r5e~q!RV;Ag&)B)s^=si2TFhA{T`G`q;4T4Ih|eNo`7qUg8U*)GNj}eIvb&^Ny~7r+cV>y;Lwhf<OoPA;%UX)Qr0tf49W=dA}t6-K%wo@D(EYDzGXz47+nv#B^SL4V9+GX>R1(-a!U(a#AVU-cZu<am^2sEcFR;2X>$**wx$zO=F}?t&J(gtY@w!7W|{j$1_~0~@dN*<wh8Huktkhbfd#2`4<^bFUhFP53vf||Bmw`tjaM*4EbjGr6O4WC4u!+oR^w*#RlJBKKL|t*wW>9SkJGH{ojQ~ktjJrXAkD}@w5ooZ_f31wJT0ga%9EvjsXQE_q^SX<q^;)aWIAjrR7PERHmFjnI3l5TiIm#KH}?V=2CRvyop{gr`CH(e8`^G)G{rpU{%!Mqd^hrLq2SK9XK#asGc_%jP91OUVmofV)RICOue4G<%5bh=b0iUZO4(we(uq4H&=9LXkXyi3>5DS0)Grep-Ig@8^X)2<qOF#hNC_g~15K3q7PQN#!QZKu$7?#+i18K3;Wu8f6|Ny{{skq~(e*V75l@x5b(u^rHgNjF;Q;G(<FT$;E9k{wXbe&!M~sB**McagLQ%yiR~8%Dc#N>{5JN`F^#PauItFtPV1SSamhrTV<}f(RawayVw$lfoG~`J#pf%dZCwPMgqD)IP!NK7+2hk6YZ-oh~vrz8gbzEGKV+W4#!1E;rBYSK-r5T#Ws~b{0iryoDPhvzN@qo7>v4T*3c2){P0Zg*i{am{iaXpt2!q+PfVD!VCLarEjNK>vE8{<pTN=mv$OT8d#(Z*&g<oJXw=tu_Z_1}zZ7e_g5d2{SJ`DWJ=!RZ1=%j_vCHGoXB<ciisFx9-c2~EDvyNAZ4X1jt}27y>4jSUbZR{13GA2J2qlW^Fxq3j$=k)UA(*6kt%*%I-IwxhFiZT%I?XDp=AJKtk9t;=28%0sM0!Hq(90LSvu5{p4evDXCG@QSI6sLFQMqz~-UrVc(!g}4Ubdy*H45~S!Jk&HElzHMmQpI~0TIQZ_}D{q~a;Hx@aTJ%vL70dE83PVyreq&Z^%788ji<uM);Z{H9#epu{!=4b&Kk4G&Jx}sn%_`NEhwp{}sm^Z=%F#fOrGl$Yj4CFu5lS9)|MdV+f|fzU3%jobULe;9wvDwe#AQiylEb_MJqXkaO?=YA1D{Odb*+0gXoz<qx@QJ;^95O{ta)lry006?4%3~%yd@)rn3xn7YX5vcbS&d?c~GXmB(3CGyY!hv)VOn#RnPWP{1kNb8}F^~oB|miOtVt4BaMmRpPE1Qa=0-TS2$TEIII$2yQMb+e!U`x2ws2lmPp=~;3}m|lYww#88z;+hLWGE^oftVv;uRw;-;$^vQw(ta^15#rY1FHBr9jn=qb=*=%gCFK5~P#X{W%^e1Uo$YkLxaox9VzVaEkpJOPL})AonJdtHvFWv5GqCi<&as`=i!uba+|pa`kX9O_)b{8!@;a_GDGgRKqae)r-RHM}A3kbv6wM^GaQ<`qKQRbwt17RpX~B_)6PSRC)*I6IZxV)5&0Z3(_pwcXs8A^pE8lDM}$we5QW?mpiA-Fn}y-EF^Z-0~gwxVluoQwpoy<P1kYVFZzPHijfND-bZp2Gxtfn>hg!&k{XJTwH~ZhE2W+$RiL9s6AU6_$ydMK17#e9%4$dE#x3u)@xG=bSiotK9oLOLhO!EjXO4PRc$l_PVOT7$nE#lrBrVRX;F&+Fn>@dq5~KGILOi3ihP6F*V5EV+WRXf^xdXIu#i_sa<7y3Keae1YFGGrBrGjh!r}_rckE?oSF?!wXkT<^D^K|Pr=mNs@lK5iXiCs{i#CqOefC&643)*H96z$0C7lEyol=6MA+-GRaoCf-l;guESOi|K5nLO`xL5mbKT5a9rEnS)fr6!K-c;<stTHAtt_7yn3)D0SrQAsAXH30}70E=}wacFL%K$A8Zq!Q*gCKrJ_xgZwrSY@l0{w%iZ<-2ArGCL(g>nUUmMqW~<m)lviOYzkL-GiRg9A0?G?aq#l@eJnN)ZGW)c93288L&gH#Da3u;%Z&V+I^>B&NV#XL16fBYGp}Voo8bd99dEx4@{>uywgROob;tUE=t4Wj<-eDf1)Q&VNLWM)!QQ-=$WA;W3?}(IOk*3f-_(&74aUy|h<o4#My9nK$h0_yyH+pbopul0{rcAMYB|Q%J-!5vlUqI>E|D&ukyX=5S<1jf6TmA~@_+jY;}1>x1wcQ4+8#8Zr-d6IJ{}{=gF}us1Yz09U!5LsbZ;2D;~fB9DVsg=7G6PEthF?un=XjZokVB2tD%k?WxHxq4T7S)$>1zNR<JPuJChyr7Oe<>~92UeM^D{8#aV_HDdEKbA`_Mn`A!bkWwA;FxXtQ&)d7)NmxUjFL~GR42j!M#`C9x|f#PzZ7V}Co+MeHor)Ja|2<Tq$s{ALna4k2l06o#zXK6`KXn=Qh=2a&H`(U0jV~ZIzsDeqLq600D&x7Qf%iwd&@vP0^;#X+*`fdE}?gfuCInBqnL!Hf;6&wX=Cp#GFjT3E<No1b*Z-IQmRSilMtQv)uX!C=vY~HI@PES1E@3sS#n_~3pa7BR)v3^jkp0X(gF-}8wzLBf&2m82wjWFKuoiSJGjZhP-}FPtR*5<Oot#btTTju?wmP_pQ2gx6Brw)zPS>=@!sHBHl@>sCjdG(Slb0mF2R5q(R@Un7%X)#5`llgIssSRlGpn(Wq%Fy)}UB2`yQ(CL&InUPF#`K>!*qxx`)GZP5iYVE%DZ{+B9~Bl6M^huHM0UdsS-+TqCOjGU&~=>rDy%KfJ&rM222&;I*)yxKQjYU-<x0AwQk(oHW@1L@S3~pieQqMQK*R0^n#{ICOh=V5iPX@BoeCa+;>%zJR`Bt8KY7ZSKZZ2!&@sbsTMnYBcTtzRe=M9|&_b%G!Kc5XYgBC<Ja|eMiO-629cMiT{!^4#RNGupLfa0&ExGgpm0IoFM|eHTr@mMf@xMRD@b8t<kxu&fzR40CH&4x#$a<o7qqT9zruFTLSuG%uP{y3Opv*O)#o43Q_Vm325MEuw79(t^hNn`>!Cc-dWp;+&G79MMA?&+&oL}1xFoS=lV7+wN-PlQ(!P8?64RI#)<;%Snv+INYJSg)RF8sAVVbSx|0oJ%7FsqtVTlj5J-MuXTi~NEZ*32*zODuFy(U?QtUyA#2!eIko8U4C(%zcy6uoM3gz3ZjOVa2ke37WDZV4Igbu$U`LWwR*lcdLFudPsT)uR$it1tg+L6+>QSJwyL10gCqQgyeA=wURx6q8JhGvUJsUey5;W(lznQi|xYmlk4Ibi9vW>mfN9njUKPh1G(1k90H*Hg=ChTMyB?5pe#j-(-U-;JoYc*$DE8alYNi4j>HS4k4KW04d9@~o4MaCF=30>Y6q1Ei~8^j32cXRU_CM+D*O)Ibop9PvJ`?OSfJW0RYUV)`gy+9e4%V%8HV4IAsIKnn&}K-nS6hVaNYci5E>T&;8AKxEpAG9b*;0y7@O(`Cem23%Moj-wzZ03-ExC_Ds$z2Cii^TYd-chTv|`_t$TZ(f{Ox&|B!`pDrpnpl>%h6(B|@q%6mjtNILG2Ma!CdEYWBueo#+2h8cE(X+zh--J-Z>7!NCI2+-x;2V2`DZ)Bbb3qd$v)k~p(Ryu#oPn^j(}|?*FL2=8Cv64tAnde497*wWP?*xcaRR<e5p?Jf`$v{l<O78#Tt4C5OkT4hgn3h26TXsTpCXO@UuX_#ne16U*y7B7_P5d%5nyc4pOS9-0({wz@fLch??iV{E}0gh~rNkoVuM&e&~&bt0>1d>Pp_jl>h_@KPZco0YW(Q`4teQ@w+=!&hL{O{)O=lEqQY4v?9`LV44WgiJ%ryS`ThUGBInd6VRW`*vIF!2+;~9V!&XUz}-u@{BTMSacrRNO0;<)kv!V%9wngSMb?0&5^Oa`YF4AAemN6#FA0RmQ)RddBq;4dMf<mRHay2xx}u^reAOguK^2m=Wr0oIDDJANXM@9Yebebl4VoY>r95kji68`mWSrEFnLQ*8hZrFJ>K)+@j&rf*-u^qz#Av&@JI`Egt86eyl=;BSm<$N;2rGc4G9I}zwn=2v0|d>^WSvG_L+hYK8mM>zC>~8#adzQL5v(SgsvG2}^)SfdOANHbK+vnWz<eeSdvk3vx{1_+YNeC|-g!G-gzTl58BG4P_{+a%OK-Nl`Ss8LSa}!Azx*fYhgQG->ECX=|Nf`{`Hx@!{I63l`}NQNoI)LVGx^KEq!6g`>!1Fs3Tt`az?FA~niRrmlR!zKjQ3iwCl|m96gLvNc|hevrEnF(D{m4B<vT^5BKTWcuaV-!Aw7dW1O!Q=l>F5G&VCM4$XZpLLPc)`2Y`y&Fd%r!CxnA$WLRZim7~B~Pb6mq5e3xHCS5x#7;C0$iMw159VDjSD^VDY9czmmR^Cm%^&tI=x7>nnw5nmV8?y6ZDdvhed^~K1&omd0_2^nrB^Ilur+SE1P@TB3@0lrg8Zm_wp<Bv3C@?RG05GD-Rr*x{Oa(T$8eb(7rk--U!B*immF<5>CNJpS4=@<T6+4qcxHH$tj5D%Vzblyy&s7?uoH<r%Hm#7vJc`X|an_J?Oe(RPOk@Y<L$eJrACrYSs{q7BASn+msAjdMz+%8|8!}5K^+j}Nd+1_nM(R`%;_Vztoq1e=gbOM@NKOcs(eBAubP`DVHYvCVsj(ofZqVdtyC>W-Q^7MuYAT4sNo9zvoe-cAUc(`)JZTQ$5(2MvyvZ8$V7g05p{K!hr?X8oUC+RZ#B$v?SoIV<q=Zm)CZs1SIOhlak<RRMM@-L@G|nmcm_u&wBtL!9>GSy95bJfH?4iuiXl>ZP8a&b9jsj9#H6J|jce!q<?ft>7Tc6UY$X@zS6}OJgZfep>kD&bETHMy+TjpkW;U-%c6XwZb(;nvQ!-IEy0m{xmJd~tz?TgT#D8H!d&_PC$tPMPhjDZBW<qhDymTgpb;UjJVPX>DK=UJ!IPCxa?MgSf~f^3|G_#uvT4v(=Nk&$Ya()_`}!{avL-wsK1B)XCig?vmg`S`2LLH_p6?Et?sh*vDVt<FO~s-J}Zw$qklhUq>KI?UFKj1jr}plQ8gMJ@bi#P_l;Ok%@U?46I=quOgV<@L_h$w!7~D#`u-QEaA>aHu<&QurOZ6`&ua**8)<%1plPdb7^^Q|nc+?XJ}H5Zf7eWM%$nG=F^W@FhqPVk98t?1A={SLZ%CBO1ysh%kX5ORykQ0wwu%1a=(u&{q~NC_)E(02adno@KUs!&;;jU3vtVs(`dpG6dYV`N{Rc8qL*_E!5pZSG!*f7`BrEV7hI4_?ILa&ck*AEYFyRL}FmrefpBP6_dT7+~T<s0ELgGhQh^tazU4O;a-o{PRHa3mu<)&UkLQMkalZWHoQmVDRr49SMYE|56nA9-r*fhTw|baHo+vV4i-z(Fq%1yH_&*1m&GP@HjdCPFeaGJO=OYhpwN;itmj1y16&^?JnR?DEnHfj@1MVkUVQiFSR^%4moWxquov0w4j5JnQNh8?#qf{6{qA{m{O$99@NGjttJ)sbMJg?*u%-R}+c&TOK2p8*+}+AO4niSHPLa@kp`!qDF3Y3Z*yAtJ-4#BK!uC4vm|B4VM2TSGdmaUR9R>Cpo;vU;j7f~&CwK)K)oYdV)x(U!r4?7+^jn5O;e{H#C0m^0$g=`Go9KM0=W|5=4l&XM{q?68s1V2a>+?Iys%QeV>?JE4a_v~a5oyT?GYDrQa!QTxKCve%GDIL+`m_n~J%<4p%lV@N3cczpCggzv<#BZ}I1GnjnTJD|XTW7pG#FOD{gU!?=@Q@Er?UB^NWsfzaMXmtuZ|kP%oh!qSy?^;ND>Y{0*mVNyG`+>^%9=(tqo!^S>sp^Wi@y{R4}eDCalOYhpS45O>lq2I}rK-Cl=IeRjh&kaTcSk#;zyI4jq8!={0EHq4FbDnTqs`|A-l{3#@Xas!-8U6C9XjK^(H`ilQR}`=X$M^(i{nm@TouXh)BYRvU2GgcM6@kvFKwel;-AdS@Os(txjM)H29RcA+g+(rP8@Q#6%o+W63fEBlc6x>0!i;?=8wfz$&(g&fRTKOG==R8pjFLg78g_2hK*WE_(=+1HtJYY_Qsg<e7eYfddyXsiu3#<aCluy#_MBYkiwgzq6tfl(osMW1BMj?;KSO?-D=DR?hU3skMum-L-rRBUkngU$HW%%6HZk^|MTw*_$+#61)T!8sg!BajBg_dj@^y@o$l0{f;$CekU8OlrwCodsmW8#+c@ZfAJ|MOf&&^Oe#W0IxgK7uyKn6VQbNok*g{DHNBIQGv)mTj96Eup?)nrk*uc0UQmodWt&uE(GkrY%Ma7k**l<!KHog#mVnpK7Sp3^X|>-)05XP#*hd+!$5~ZxS;~xUmZu!U%eXp$G#Gw%=m5e8srl*fap-Vhm^_#)ahs}T_E%qztHzE;K_X98DPq4h9Oi&&9^V#o=`d5h0OBzr!U@oce+O^VU#AN$_I6Cur6a?s!`~oMH&F{<(oa=0&2$w();z~#ExYYewcjBvp6+NkhQ?b@Y@rhas-K&Vx9fiZw{EeETK{hHNqPqPXy{SI!$EvaGOD-5PBl9vxC4s3*7wLy&brLV0);@++$x@n*sSb5fj~o`O+tNfOAw}iG~~2=)6IjCS9Hp8}x;C0a?l5=buqS1z~t~Ho_90gOSwE95=s}6&a3oIGu6p^A_AFme<@GXT%mU&V#bZ`+eELdFvM0JEwN*mg?Lodw0{=S?wF<(XcHx9mF3pLXpn^n401O*mdC_%kt5W<-_2|_h0k?yh5u@IIS_~y1u4n@;wq!F{*3pDJ19Q;86I8z=aa@8-7Q4x}-BHLGW`&E?q~O(U#zguD$j;r{i0arzSZhTHIuAfymevWsYh!`2I{K2VK_BV#n6wN7{Sj!Z#RX`G89Yc(Yu?+Ed8tLkaf0Zzx6Tmnp|3?VKEK#6$D;B?T&xQ24w{22(uE+0vQgrGum4@eaAf;Og5U)wJ|w8(EsyNV4ns_gCHutFx`VRbL#ukJh&LNiVAUV-p_-tajtoq2398$waDl?NINBX2c3dNAuA;0ozG&qavoY#d?fE++9J__fd&BTmz&RfbtONb{`PXL;9TARDA*Uy?Er&v3tT;!wdygz7cOEt#ic?JT)UJ3Iwn_F=!KIPxyMkmA|7dd}tDBdN4~a2jy}-*plbpjz;1emC$CgU4YBNVz5a!I~xg0`2aJRj1P~V4EtXVhlXdAUQmPT+hAekfmnUqi;XP#=xt^wd5Ez?|BsIRdjJRKR>IjZ$*+y!S>)vaB3?u?9XdM`SgmUK(GC_+6KH*b#$`W&Qu#ShSrgqcrtv18q-5js7dc3aAQvL)QT<Mila^H*SQN<p>eD~^(s1U#KZ%Z?AAfriy?uWAt@3)neZYoQ+_<5n+=wreIuDT><mt;loPfL`I2%ZHzWi#CAoH=LaBw17pk-l$QjapdvfRwHP6BhQaI!&3I=Z-m4<r$fK&y+;7LQEH>-gQt^V5^)^_%F2m#<&E`QffX6JtdaLW8OtI>j&?+b`hhjQOwp-g7zTzfLEdkMOB!_)JKr*CeCF#5FI}-Hg}})g&{%wGpkV>Tq?*xg-VRrN(r0{GMyp*UXMH?{>ap6Iu?MTxcY|A=Q8<sHAH|T!Yi8`6hNTfXjF^@m9wYEY+4BO=WcTHiNtWg;ZcTJPddq<`K}5x|$~ddIKvsjStm^V<YVGRk~PK^<(g%q&uyu|6aK1New+Ix8;YFN*SGtI6!4RdY>y}9BM5LGo0K6I#kL<*5v2lKB?95Y7W%xHbxG&`g`2`5mJIQP?LXlAc~)%kI&aUK@y^#9Z-?)bodat&hY@K!SnAg45~Kbqj$4Zm=uwo!X09jnM2rbEU4OqIu4cPLiY!&?rrP;@R8K^oF13{y0x%mLq>*OP6FbiNF5W|XQ+)`*)>>4qfbBbdr^!)+f!7HU{JK@l8MEAv~~spWeF)#eUPg8WhX`EZof&%U+nxzZkgIlpY%b(we89{KEx77@};ACVk3ijJ%?=t5)6Ed8iWn}Sd1q{S;f%AH;$fqLWU^G7r8{i9k2C-F2&F|cx#8v5kGN<q@c1N)w0j16i1MLrZXr_h9fj`A~n;{D5hZ~u_zm%zA6Z^ew(t>*R==wnrVa^i4??v^{JSBEsMZ-SAH#9C1g>T^;9D>hq$zx7fN5Y+y4{(u#Y(AU};J1^Je$l;jN+ViHOkJH{C$n7dyLBbL{gq#AS+~bH8a()w(`ex$9GvI@WTph=Nw~+DAKTF+OKVLd{LzfeQ8Un#L!)*aL*~uG2TUbdW9i)u5Y-NJLMuhNb(<TFSM=;WKA&I=mR|_~28EKGXQLF@6M^(`K39g6?&G1*Eoe<(N^UuqdLR#^^G{uet&Lb$k)`P@;W6qS5`5p}1N*(p$&@QTh`*w~)@2QqPWKWk;!d2;VzFN}KbcnIoko$ZfAQDK!EVsqEV1rW*JBU($DdLj'


import base64
import hashlib
import importlib.util
import zlib

def sha256(body):
    return hashlib.sha256(body).hexdigest()

sample_bytes = ''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in SAMPLES).encode('utf-8')
crawler_bytes = zlib.decompress(base64.b85decode(CRAWLER_PAYLOAD))
if sha256(crawler_bytes) != CRAWLER_SHA256:
    raise ValueError('Snapshot crawler bị hỏng.')
if (len(SAMPLES) != EXPECTED_URL_COUNT or len({r['domain'] for r in SAMPLES}) != EXPECTED_DOMAIN_COUNT
        or len({r['url'] for r in SAMPLES}) != len(SAMPLES) or len({r['id'] for r in SAMPLES}) != len(SAMPLES)):
    raise ValueError('Danh sách mẫu thiếu hoặc trùng ID/URL.')
manifest_path = RUN_DIR / 'manifest.json'
sample_path = RUN_DIR / 'sample.jsonl'
crawler_path = RUN_DIR / 'crawler_snapshot.py'
if manifest_path.exists():
    saved = json.loads(manifest_path.read_text(encoding='utf-8'))
    if (saved.get('probe_kind') != PROBE_KIND
            or saved.get('sample_sha256') != sha256(sample_bytes)
            or saved.get('source_sha256') != SOURCE_SHA256):
        raise ValueError('Run cũ không khớp mẫu của notebook này.')
    if not sample_path.exists() or sample_path.read_bytes() != sample_bytes:
        raise ValueError('Danh sách mẫu của run cũ đã thay đổi.')
    if not crawler_path.exists() or sha256(crawler_path.read_bytes()) != CRAWLER_SHA256:
        raise ValueError('Snapshot khác run cũ; hãy tạo run mới.')
    if json.loads((RUN_DIR / 'source_profile.json').read_text(encoding='utf-8')) != SOURCE_PROFILE:
        raise ValueError('Source profile của run cũ đã thay đổi.')
else:
    sample_path.write_bytes(sample_bytes)
    crawler_path.write_bytes(crawler_bytes)
    (RUN_DIR / 'source_profile.json').write_text(
        json.dumps(SOURCE_PROFILE, ensure_ascii=False, indent=2), encoding='utf-8')
    manifest_path.write_text(json.dumps({
        'created_at': datetime.now(timezone.utc).isoformat(),
        'probe_kind': PROBE_KIND, 'dataset': 'AIGuruTinix/ViBioMIR',
        'source': 'embedded samples from data/links_corpus.parquet',
        'source_sha256': SOURCE_SHA256, 'source_rows': SOURCE_PROFILE['rows'],
        'sampling': SAMPLING_METHOD, 'seed': 42, 'selected': len(SAMPLES),
        'selected_domains': EXPECTED_DOMAIN_COUNT,
        'baseline_archive': BASELINE_ARCHIVE, 'baseline_archive_sha256': BASELINE_ARCHIVE_SHA256,
        'sample_sha256': sha256(sample_bytes),
    }, ensure_ascii=False, indent=2), encoding='utf-8')

spec = importlib.util.spec_from_file_location('domain_probe_crawler', crawler_path)
crawler = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = crawler
spec.loader.exec_module(crawler)
priority_path = RUN_DIR / 'priority_probe_snapshot.py'
priority_bytes = zlib.decompress(base64.b85decode(PRIORITY_PAYLOAD))
if sha256(priority_bytes) != PRIORITY_SHA256:
    raise ValueError('Priority probe snapshot bị hỏng.')
if priority_path.exists() and sha256(priority_path.read_bytes()) != PRIORITY_SHA256:
    raise ValueError('Priority probe khác run cũ; tạo run mới.')
priority_path.write_bytes(priority_bytes)
priority_spec = importlib.util.spec_from_file_location('domain_priority_probe', priority_path)
priority_probe = importlib.util.module_from_spec(priority_spec)
sys.modules[priority_spec.name] = priority_probe
priority_spec.loader.exec_module(priority_probe)
if not isinstance(USER_AGENT, str) or not USER_AGENT.strip() or any(c in USER_AGENT for c in ('\r', '\n')):
    raise ValueError('USER_AGENT phải là chuỗi header hợp lệ.')
crawler.UA = USER_AGENT
crawler.ROBOT_AGENT = USER_AGENT.split('/', 1)[0]
(RUN_DIR / 'baseline_domains.json').write_text(json.dumps(BASELINE_DOMAINS, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Đã nạp {len(SAMPLES)} URL mẫu và crawler {crawler.VERSION}.')


## 4. Chuẩn bị bảng kết quả và phân loại lỗi

Output riêng: `vao_duoc.csv`, `khong_vao_duoc.csv`, `chua_chay.csv`,
`loi_theo_loai.csv`, `can_xem_noi_dung.csv`; bảng tổng là `ket_qua.csv`.
Các CSV dùng UTF-8 BOM để mở được tiếng Việt trong Excel.
`error_detail` giữ nguyên lỗi kỹ thuật; `reason` diễn giải tiếng Việt.
Mở `ket_qua.html` để lọc theo domain hoặc trạng thái.
`uu_tien_fix_domain.csv/html` xếp hostname còn lỗi theo số URL dataset giảm dần.
`tong_hop_domain.csv` gồm cả hostname đã phục hồi; `WEB_ERRORS.md` là báo cáo lỗi chi tiết.


In [ ]:
import csv
import html
import shutil
from collections import Counter
from IPython.display import HTML, FileLink, display

STATUS_REASONS = {
    'robots_unknown': 'Không xác định được chính sách robots.txt; crawler dừng trước khi tải trang.',
    'robots_denied': 'robots.txt trả HTTP 401/403; crawler dừng theo chính sách robots.',
    'robots_disallowed': 'robots.txt không cho phép crawler tải URL này.',
    'http_error': 'Trang đích trả HTTP ngoài khoảng 200–299.',
    'network_error': 'Không tải được trang do lỗi kết nối sau retry.',
    'unsafe_address': 'Địa chỉ đích không phải IP public được crawler cho phép.',
    'invalid_url': 'URL không hợp lệ hoặc dùng cổng ngoài 80/443.',
    'oversized': 'Phản hồi vượt giới hạn kích thước của crawler.',
    'response_deadline': 'Tải phản hồi vượt giới hạn thời gian tổng.',
    'rate_limited_deferred': 'Server yêu cầu chờ Retry-After quá lâu; crawler hoãn URL.',
    'redirect_limit': 'Vượt số lần redirect cho phép.',
    'unsupported_content_type': 'Đã tải phản hồi nhưng định dạng không được hỗ trợ.',
    'unexpected_error': 'Lỗi ngoài dự kiến; xem error_detail.',
}

def classify_error(status, detail, http_status):
    lower = detail.lower()
    if status == 'http_error':
        return f'page_http_{http_status}'
    if status == 'robots_disallowed':
        return 'robots_disallowed'
    if status.startswith('robots_') and 'robots http ' in lower:
        return 'robots_http_' + lower.split('robots http ', 1)[1].split()[0]
    if 'html instead of robots.txt' in lower:
        return 'robots_html_instead_of_txt'
    prefix = 'robots_' if status.startswith('robots_') else ''
    if 'waiting for domain lock' in lower:
        return prefix + 'domain_lock_budget'
    if 'request budget exhausted' in lower or 'retry budget exhausted' in lower:
        return prefix + 'budget_exhausted'
    if 'winerror 10013' in lower or 'network is unreachable' in lower:
        return prefix + 'environment_network_blocked'
    if any(s in lower for s in ('timed out', 'timeout')):
        return prefix + 'timeout'
    if any(s in lower for s in ('sslerror', 'certificate verify failed', 'ssl:')):
        return prefix + 'ssl_error'
    if any(s in lower for s in ('nameresolutionerror', 'getaddrinfo failed', 'name or service not known', 'failed to resolve')):
        return prefix + 'dns_error'
    if any(s in lower for s in ('connection refused', 'failed to establish a new connection', 'connection reset')):
        return prefix + 'connection_error'
    return status

def result_rows():
    rows = []
    for sample in sorted(SAMPLES, key=lambda s: (-SOURCE_PROFILE['host_counts'][s['domain']], s['domain'], s['sample_index'])):
        record_path = RUN_DIR / 'records' / f"{sample['id']}.json"
        record = json.loads(record_path.read_text(encoding='utf-8')) if record_path.exists() else {}
        if record and (record.get('id'), record.get('url')) != (sample['id'], sample['url']):
            raise ValueError('Checkpoint không khớp ID/URL nguồn.')
        status = record.get('status', 'pending')
        page_http = record.get('http_status')
        robots = record.get('robots', {})
        # Robots failure may happen before fetch() attaches its evidence to the record.
        if status.startswith('robots_') and not robots:
            from urllib.parse import urlsplit
            target = record.get('requests', [{}])[-1].get('url', sample['url']) if record.get('requests') else sample['url']
            if record.get('redirects'):
                target = record['redirects'][-1]['to']
            p = urlsplit(target)
            robot_file = RUN_DIR / 'robots' / (sha256(f'{p.scheme}://{p.netloc}'.encode()) + '.json')
            if robot_file.exists():
                robots = json.loads(robot_file.read_text(encoding='utf-8'))
        detail = record.get('error', '')
        browser_file = RUN_DIR / 'browser_records' / f"{sample['id']}.json"
        browser = json.loads(browser_file.read_text(encoding='utf-8')) if browser_file.exists() else {}
        if browser and (browser.get('id'), browser.get('url')) != (sample['id'], sample['url']):
            raise ValueError('Browser checkpoint không khớp ID/URL.')
        requests_access = ('CHUA_CHAY' if status == 'pending' else
                  'VAO_DUOC' if page_http is not None and 200 <= page_http < 300 else 'KHONG_VAO_DUOC')
        if DIRECT_PUBLIC_PROBE:
            access, effective_engine, effective = priority_probe.access_outcome(record, browser)
            if record.get('challenge_detected'):
                requests_access = 'KHONG_VAO_DUOC'
        else:
            access, effective_engine, effective = requests_access, 'requests', record
        flags = record.get('quality_flags', [])
        if status == 'pending':
            reason, error_type, stage = 'Chưa chạy URL này.', '', 'pending'
        elif status == 'extracted':
            stage = 'done'
            error_type = ('qc_' + record['quality_status'] if record.get('quality_status') in ('review', 'rejected') else '')
            reason = ('Tải được và nội dung đạt QC tự động.' if record.get('quality_status') == 'accepted'
                      else 'Tải được, nhưng nội dung cần xem lại hoặc bị QC loại: ' + ', '.join(flags))
        else:
            stage = ('robots' if status.startswith('robots_') else 'extraction' if access == 'VAO_DUOC' else 'fetch')
            error_type = record.get('error_kind') or classify_error(status, detail, page_http)
            reason = ('Trang trả HTML xác minh/JavaScript thay vì nội dung bài viết.'
                      if record.get('challenge_detected') else STATUS_REASONS.get(status, status))
        requests_reason = reason
        if effective_engine == 'selenium':
            stage = 'done_browser'
            reason = 'Selenium cải thiện tải/trích xuất so với Requests. QC: ' + effective.get('quality_status', 'unknown')
        rows.append({
            'domain': sample['domain'], 'dataset_url_count': SOURCE_PROFILE['host_counts'][sample['domain']],
            'sample_id': sample['id'], 'sample_url': sample['url'], 'access_result': access,
            'requests_access_result': requests_access, 'effective_engine': effective_engine,
            'effective_http_status': effective.get('http_status'),
            'effective_quality_status': effective.get('quality_status', ''),
            'effective_char_count': effective.get('char_count'),
            'effective_quality_flags': '; '.join(effective.get('quality_flags', [])),
            'http_status': page_http, 'robots_http_status': robots.get('status'),
            'robots_attempt_count': robots.get('attempt_count'),
            'robots_cache_hit': robots.get('cache_hit'),
            'robots_fetch_id': robots.get('fetch_id', ''),
            'robots_error_kind': robots.get('error_kind', ''),
            'robots_elapsed_seconds': robots.get('elapsed_seconds'),
            'robots_lock_wait_seconds': robots.get('lock_wait_seconds'),
            'crawl_status': status, 'failure_stage': stage, 'error_type': error_type,
            'reason': reason, 'requests_reason': requests_reason,
            'error_detail': detail, 'quality_status': record.get('quality_status', ''),
            'quality_flags': '; '.join(flags), 'char_count': record.get('char_count'),
            'title': record.get('title', ''), 'final_url': record.get('final_url', ''),
            'elapsed_seconds': record.get('elapsed_seconds'),
            'reused_from': record.get('reused_from', ''),
            'request_attempt_count': len(record.get('requests', [])),
            'retryable': record.get('retryable'), 'retry_deferred': record.get('retry_deferred', ''),
            'page_attempted': bool(record.get('requests')),
            'browser_status': browser.get('status', ''),
            'browser_http_status': browser.get('http_status'),
            'browser_quality_status': browser.get('quality_status', ''),
            'browser_char_count': browser.get('char_count'),
            'browser_final_url': browser.get('final_url', ''),
            'browser_error': browser.get('error', ''),
            'browser_attempt_count': browser.get('attempt_count'),
            'sample_role': sample['sample_role'], 'sample_index': sample['sample_index'],
        })
    return rows

def domain_rows(rows):
    result = []
    for domain in sorted(BASELINE_DOMAINS, key=lambda h: (-SOURCE_PROFILE['host_counts'][h], h)):
        selected = [r for r in rows if r['domain'] == domain]
        failed = [r for r in selected if r['access_result'] == 'KHONG_VAO_DUOC']
        content = [r for r in selected if r['access_result'] == 'VAO_DUOC' and r['effective_quality_status'] != 'accepted']
        pending = sum(r['access_result'] == 'CHUA_CHAY' for r in selected)
        errors = Counter(r['error_type'] for r in failed)
        errors.update('qc_' + (r['effective_quality_status'] or 'unknown') for r in content)
        result.append({
            'domain': domain, 'dataset_url_count': SOURCE_PROFILE['host_counts'][domain],
            'baseline_error_type': BASELINE_DOMAINS[domain]['error_type'],
            'selected_urls': len(selected), 'additional_urls': sum(r['sample_role'] == 'additional' for r in selected),
            'missing_additional_urls': BASELINE_DOMAINS[domain]['shortfall'],
            'completed_urls': len(selected) - pending, 'pending_urls': pending,
            'page_not_attempted': sum(not r['page_attempted'] and r['access_result'] != 'CHUA_CHAY' for r in selected),
            'accessible_urls': sum(r['access_result'] == 'VAO_DUOC' for r in selected),
            'access_failures': len(failed), 'content_issues': len(content),
            'total_problem_urls': len(failed) + len(content),
            'additional_problem_urls': sum(r['sample_role'] == 'additional' for r in failed + content),
            'error_types': '; '.join(f'{kind}: {n}' for kind, n in sorted(errors.items())),
            'test_status': 'CHUA_CHAY' if pending == len(selected) else 'CHUA_HOAN_TAT' if pending else 'CON_LOI' if failed or content else 'DA_PHUC_HOI',
        })
    return result

def write_csv(name, rows, fields):
    temporary = RUN_DIR / (name + '.tmp')
    with temporary.open('w', encoding='utf-8-sig', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    temporary.replace(RUN_DIR / name)

def save_outputs(show=True):
    rows = result_rows()
    fields = list(rows[0])
    counts = Counter(r['access_result'] for r in rows)
    summary = {'selected': len(rows), 'vao_duoc': counts['VAO_DUOC'],
               'khong_vao_duoc': counts['KHONG_VAO_DUOC'], 'chua_chay': counts['CHUA_CHAY'],
               'crawl_status_counts': dict(Counter(r['crawl_status'] for r in rows)),
               'quality_status_counts': dict(Counter(r['quality_status'] for r in rows if r['quality_status'])),
               'effective_quality_status_counts': dict(Counter(r['effective_quality_status'] for r in rows if r['effective_quality_status'])),
               'created_at': datetime.now(timezone.utc).isoformat()}
    domains = domain_rows(rows)
    priority = [dict(priority_rank=i, **r) for i, r in enumerate((r for r in domains if r['total_problem_urls']), 1)]
    summary.update(hostnames=len(domains), domains_with_errors=len(priority),
                   priority_order='dataset_url_count descending', user_agent=USER_AGENT,
                   page_not_attempted=sum(not r['page_attempted'] and r['access_result'] != 'CHUA_CHAY' for r in rows),
                   additional_urls=sum(r['sample_role'] == 'additional' for r in rows))
    summary.update(request_mode='direct_public_page_probe' if DIRECT_PUBLIC_PROBE else 'robots_enforced',
                   requests_http_2xx=sum(r['requests_access_result'] == 'VAO_DUOC' for r in rows),
                   browser_recovered=sum(r['access_result'] == 'VAO_DUOC' and r['effective_engine'] == 'selenium' for r in rows),
                   browser_access_recovered=sum(r['access_result'] == 'VAO_DUOC' and r['effective_engine'] == 'selenium' and r['requests_access_result'] != 'VAO_DUOC' for r in rows),
                   browser_content_improved=sum(r['access_result'] == 'VAO_DUOC' and r['effective_engine'] == 'selenium' and r['requests_access_result'] == 'VAO_DUOC' for r in rows),
                   browser_tested=sum(bool(r['browser_status']) for r in rows),
                   browser_http_2xx=sum(r['browser_http_status'] is not None and 200 <= r['browser_http_status'] < 300 for r in rows))
    write_csv('tong_hop_domain.csv', domains, list(domains[0]))
    write_csv('uu_tien_fix_domain.csv', priority, ['priority_rank', *domains[0]])
    (RUN_DIR / 'uu_tien_fix_domain.json').write_text(json.dumps(priority, ensure_ascii=False, indent=2), encoding='utf-8')
    write_csv('ket_qua.csv', rows, fields)
    write_csv('requests_vs_selenium.csv', rows, fields)
    for value, filename in [('VAO_DUOC', 'vao_duoc.csv'), ('KHONG_VAO_DUOC', 'khong_vao_duoc.csv'), ('CHUA_CHAY', 'chua_chay.csv')]:
        write_csv(filename, [r for r in rows if r['access_result'] == value], fields)
    write_csv('can_xem_noi_dung.csv', [r for r in rows if r['access_result'] == 'VAO_DUOC' and r['effective_quality_status'] != 'accepted'], fields)
    failures = Counter(r['error_type'] for r in rows if r['access_result'] == 'KHONG_VAO_DUOC')
    errors = [{'error_type': kind, 'domain_count': len({r['domain'] for r in rows if r['access_result'] == 'KHONG_VAO_DUOC' and r['error_type'] == kind}),
               'url_count': n,
               'domains': '; '.join(sorted({r['domain'] for r in rows if r['access_result'] == 'KHONG_VAO_DUOC' and r['error_type'] == kind}))}
              for kind, n in failures.most_common()]
    write_csv('loi_theo_loai.csv', errors, ['error_type', 'domain_count', 'url_count', 'domains'])
    (RUN_DIR / 'ket_qua.json').write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding='utf-8')
    (RUN_DIR / 'summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
    md = ['# Các website còn lỗi sau lượt test bổ sung', '',
          f"Thời điểm UTC: {summary['created_at']}. Run: `{RUN_DIR.name}`.", '',
          f"Đã chạy {len(rows)-counts['CHUA_CHAY']}/{len(rows)} URL thuộc {len(domains)} hostname. "
          f"Tải được sau Requests/Selenium: {counts['VAO_DUOC']}; chưa tải được: {counts['KHONG_VAO_DUOC']}; chưa chạy: {counts['CHUA_CHAY']}.", '',
          f"Requests tải được: {summary['requests_http_2xx']}; chọn kết quả Selenium: {summary['browser_recovered']} "
          f"(phục hồi truy cập: {summary['browser_access_recovered']}; cải thiện nội dung: {summary['browser_content_improved']}). "
          'Các cột lỗi Requests vẫn giữ nguyên dù Selenium phục hồi thành công.', '',
          f"{summary['page_not_attempted']} URL đã xử lý nhưng chưa request bài viết vì dừng trước bước tải bài. "
          'Các trường hợp này được phân biệt với bài viết thực sự trả HTTP lỗi.', '',
          f'HTTP User-Agent: `{USER_AGENT}`.', '',
          f"Bộ mẫu có {len(rows)} URL thuộc {len(domains)} hostname, gồm {summary['additional_urls']} URL bổ sung. "
          'Gồm lỗi truy cập và lỗi nội dung/QC. Số URL dataset là số dòng của hostname, chưa loại URL trùng.', '',
          'Ưu tiên theo số URL dataset giảm dần; không suy ra toàn bộ URL của domain đều bị lỗi. '
          'Dừng ở robots là chưa request bài viết, không đồng nghĩa website không mở được trong trình duyệt.', '',
          '| Ưu tiên | Hostname | Số URL dataset | Mẫu đã chạy | Lỗi truy cập | Lỗi nội dung | Loại lỗi |',
          '|---:|---|---:|---:|---:|---:|---|']
    for item in priority:
        md.append(f"| {item['priority_rank']} | {item['domain']} | {item['dataset_url_count']:,} | {item['completed_urls']}/{item['selected_urls']} | {item['access_failures']} | {item['content_issues']} | {item['error_types']} |")
    md += ['', '## Domain đã phục hồi trong các mẫu', '']
    md += [f"- {r['domain']}: {r['accessible_urls']}/{r['selected_urls']} URL tải được và đạt QC." for r in domains if r['test_status'] == 'DA_PHUC_HOI'] or ['Chưa xác định domain phục hồi hoàn toàn.']
    md += ['', '## Domain thiếu URL bổ sung', '']
    md += [f"- {r['domain']}: lấy được {r['additional_urls']}/4 URL mới; dataset có {r['dataset_url_count']} dòng." for r in domains if r['missing_additional_urls']]
    md += ['', '## Chi tiết lỗi theo domain', '']
    for item in priority:
        md += [f"### {item['priority_rank']}. {item['domain']}", '',
               f"{item['dataset_url_count']:,} URL nguồn; {item['page_not_attempted']} mẫu đã xử lý nhưng chưa request bài viết. "
               f"Lỗi baseline: `{item['baseline_error_type']}`.", '']
        for row in rows:
            if row['domain'] != item['domain'] or row['access_result'] == 'CHUA_CHAY' or (row['access_result'] == 'VAO_DUOC' and row['effective_quality_status'] == 'accepted'):
                continue
            detail = (row['error_detail'] or row['quality_flags']).replace('\n', ' ').replace('`', "'")
            md += [f"- ID {row['sample_id']} ({row['sample_role']}): [URL](<{row['sample_url']}>) — `{row['error_type']}`. "
                   f"HTTP trang: {row['http_status']}; HTTP robots: {row['robots_http_status']}. {row['reason']} Lỗi gốc: `{detail}`."]
        md.append('')
    (RUN_DIR / 'WEB_ERRORS.md').write_text('\n'.join(md), encoding='utf-8')

    visible_fields = ['domain', 'dataset_url_count', 'sample_id', 'sample_url', 'sample_role', 'access_result',
                      'http_status', 'robots_http_status', 'robots_attempt_count', 'robots_cache_hit', 'error_type', 'reason', 'error_detail',
                      'quality_status', 'quality_flags', 'char_count']
    labels = ['Domain', 'Số URL nguồn', 'ID mẫu', 'URL mẫu', 'Mẫu cũ/mới', 'Vào được?', 'HTTP trang',
              'HTTP robots', 'Lần thử robots', 'Cache hit robots', 'Loại lỗi', 'Lý do', 'Lỗi gốc', 'QC', 'Cờ QC', 'Số ký tự']
    if DIRECT_PUBLIC_PROBE:
        visible_fields = ['domain', 'dataset_url_count', 'sample_id', 'sample_url', 'http_status',
                          'error_type', 'error_detail', 'quality_status', 'char_count',
                          'browser_status', 'browser_http_status', 'browser_quality_status', 'browser_char_count', 'browser_error',
                          'access_result', 'effective_engine']
        labels = ['Domain', 'Số URL nguồn', 'ID', 'URL', 'HTTP Requests', 'Lỗi Requests', 'Chi tiết',
                  'QC Requests', 'Ký tự Requests', 'Selenium', 'HTTP Selenium', 'QC Selenium', 'Ký tự Selenium', 'Lỗi Selenium',
                  'Kết quả cuối', 'Engine tải được']
    def esc(value):
        return html.escape(str(value if value is not None else ''), quote=True)
    body = []
    for row in rows:
        cells = []
        for field in visible_fields:
            value = row[field]
            text = (f'<a href="{esc(value)}" target="_blank" rel="noopener noreferrer">{esc(value)}</a>'
                    if field == 'sample_url' else esc(value))
            cells.append('<td>' + text + '</td>')
        body.append(f'<tr data-access="{row["access_result"]}">' + ''.join(cells) + '</tr>')
    style = """<style>body{font:14px/1.5 system-ui,sans-serif;margin:20px;color:#172b4d}
    .table-wrap{overflow:auto;max-height:70vh;border:1px solid #ccd4df}table{border-collapse:collapse;width:100%}
    th,td{padding:9px;border-bottom:1px solid #dce2e9;text-align:left;vertical-align:top}th{background:#e8eef6;position:sticky;top:0}
    td{min-width:95px;max-width:330px;overflow-wrap:anywhere}td:nth-child(4),td:nth-child(9),td:nth-child(10){min-width:260px}
    input,select{padding:8px;margin:0 10px 14px 0}a{color:#1654a0}</style>"""
    table = '<div class="table-wrap"><table><thead><tr>' + ''.join('<th>' + esc(label) + '</th>' for label in labels)
    table += '</tr></thead><tbody>' + ''.join(body) + '</tbody></table></div>'
    heading = f'<h1>Kết quả thử crawl {len(rows)} URL / {len(domains)} hostname</h1><p>Vào được: {counts["VAO_DUOC"]} · Không vào được: {counts["KHONG_VAO_DUOC"]} · Chưa chạy: {counts["CHUA_CHAY"]}</p>'
    heading += '<p>Vào được tính cả Requests và Selenium phục hồi; trang challenge được loại. Các cột Requests vẫn giữ lỗi gốc, QC là kết quả riêng. Kết quả chỉ áp dụng cho các URL mẫu.</p>'
    controls = '<input id="search" placeholder="Tìm domain, URL hoặc lỗi" aria-label="Tìm kiếm"><select id="access" aria-label="Kết quả"><option value="">Tất cả</option><option>VAO_DUOC</option><option>KHONG_VAO_DUOC</option><option>CHUA_CHAY</option></select><span id="count"></span>'
    priority_fields = ['priority_rank', 'domain', 'dataset_url_count', 'selected_urls', 'completed_urls', 'pending_urls', 'access_failures', 'content_issues', 'error_types']
    priority_labels = ['Ưu tiên', 'Domain', 'Số URL dataset', 'Mẫu', 'Đã chạy', 'Chưa chạy', 'Lỗi truy cập', 'Lỗi nội dung', 'Loại lỗi']
    priority_heading = '<h2>Domain còn lỗi — số URL dataset giảm dần</h2><p>Gồm lỗi truy cập và lỗi nội dung/QC. Số URL nguồn không phải số URL đã xác minh lỗi.</p>'
    priority_table = '<div class="table-wrap"><table id="priority"><thead><tr>' + ''.join('<th>'+esc(label)+'</th>' for label in priority_labels)
    priority_table += '</tr></thead><tbody>' + ''.join('<tr>' + ''.join('<td>'+esc(r[f])+'</td>' for f in priority_fields) + '</tr>' for r in priority) + '</tbody></table></div>'
    (RUN_DIR / 'uu_tien_fix_domain.html').write_text('<!doctype html><html lang="vi"><meta charset="utf-8"><title>Ưu tiên domain lỗi</title>'+style+priority_heading+priority_table+'</html>', encoding='utf-8')
    javascript = """<script>const rs=[...document.querySelectorAll('tbody tr')],s=document.getElementById('search'),a=document.getElementById('access');
    function filter(){let n=0;for(const r of rs){const ok=r.textContent.toLowerCase().includes(s.value.toLowerCase())&&(!a.value||r.dataset.access===a.value);r.hidden=!ok;n+=ok?1:0}document.getElementById('count').textContent=n+' hostname'}
    s.addEventListener('input',filter);a.addEventListener('change',filter);filter();</script>"""
    javascript = javascript.replace("document.querySelectorAll('tbody tr')", "document.querySelectorAll('tr[data-access]')").replace("n+' hostname'", "n+' URL'")
    (RUN_DIR / 'ket_qua.html').write_text('<!doctype html><html lang="vi"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Kết quả crawl domain</title>' + style + heading + priority_heading + priority_table + '<h2>Từng URL mẫu</h2>' + controls + table + javascript + '</html>', encoding='utf-8')
    # Archive outside the run directory to avoid including the archive itself.
    zip_path = Path(shutil.make_archive(str(RUN_DIR), 'zip', root_dir=RUN_DIR.parent, base_dir=RUN_DIR.name))
    if show:
        print(json.dumps(summary, ensure_ascii=False, indent=2))
        display(HTML(style + priority_heading + priority_table))
        display(HTML(style + heading + table))
        print('File kết quả:', RUN_DIR)
        print('ZIP:', zip_path)
    return rows, summary, zip_path

# Create reports before networking, so partial runs have explicit CHUA_CHAY rows.
rows, summary, zip_path = save_outputs(show=False)
print('Đã chuẩn bị báo cáo và phân loại lỗi.')


## 5. Tải trang và đối chiếu Selenium

Chạy Requests trước, từng URL tuần tự trong một domain; các domain chạy song song.
Mỗi URL có timeout connect/read và ngân sách riêng, không tiêu hao ngân sách vì các mẫu cùng domain chờ nhau.
Ngân sách được kiểm tra giữa các bước; DNS và thử nhiều IP có thể làm thời gian thực vượt mức đặt.
Theo `BROWSER_TEST_MODE`, Selenium thử lại URL Requests lỗi hoặc HTTP 200 nhưng text rỗng/QC lỗi;
có thể đặt `off` để chỉ chạy Requests. Nếu bật Selenium, Chrome được kiểm tra trước lượt Requests.
Trên Linux, notebook tự cài thư viện Chrome còn thiếu bằng apt (gồm ATK); Selenium Manager tìm/tải
Chrome và driver. Preflight thất bại sẽ dừng ô crawl, lưu `browser_setup_error.json`,
`browser_runtime.json`, `browser_preflight.json` và log Chrome để chẩn đoán.
Requests và Selenium có kết quả riêng; DOM trống/challenge không được ghi thành tải được bài.
Requests retry tối đa 3 lần; lỗi mạng được đóng pool kết nối cũ trước lần thử tiếp theo.
Selenium dùng User-Agent thực và retry timeout tối đa 2 lần; 403/challenge không được retry liên tục.
`access_result` tính cả trường hợp Selenium phục hồi, còn các cột lỗi Requests vẫn giữ nguyên.
Kết quả cuối chọn engine có QC tốt hơn; kết quả tương đương giữ Requests.
Parser 120ask lấy câu hỏi/câu trả lời; cnkang lấy thân bài, bỏ menu/bài liên quan.
`summary.json` tách số URL phục hồi truy cập và số URL cải thiện nội dung nhờ Selenium.
Để thử lại riêng nhóm lỗi, đặt `RECOVERY_RUN_DIR` là đường dẫn run cũ và giữ `RESUME_RUN_DIR = None`.
Recovery tạo run mới, kiểm tra hash raw 2xx trước khi reuse, để các checkpoint lỗi được tải lại.
Để chạy lại toàn bộ bằng cấu hình mới, giữ `RESUME_RUN_DIR = None` và Run All.

Mặc định `failed_only` tự chuyển mọi lỗi Requests/trích xuất, text rỗng và QC `review`/`rejected`
sang Selenium, kể cả PDF cần kiểm tra lại. `off` chủ động tắt đối chiếu Selenium;
robots bị chặn và URL không hợp lệ/địa chỉ không được phép vẫn được giữ nguyên.


In [ ]:
import argparse
import os
import socket
import time

# DNS check is diagnostic only; failure here does not stop the crawl.
network_path = RUN_DIR / 'network_check.json'
if not network_path.exists():
    network = {
        'checked_at': datetime.now(timezone.utc).isoformat(),
        'environment': 'kaggle' if Path('/kaggle/working').is_dir() else 'colab' if Path('/content').is_dir() else 'local',
        'python': sys.version, 'platform': sys.platform,
        'proxy_environment_present': {key: bool(os.environ.get(key)) for key in
                                      ('HTTP_PROXY', 'HTTPS_PROXY', 'ALL_PROXY', 'NO_PROXY')},
        'dns': [],
    }
    for domain in NETWORK_CHECK_DOMAINS:
        start = time.monotonic()
        entry = {'domain': domain}
        try:
            entry['addresses'] = sorted({r[4][0] for r in socket.getaddrinfo(domain, 443)})
        except OSError as exc:
            entry.update(exception_class=type(exc).__name__, error=str(exc))
        entry['elapsed_seconds'] = round(time.monotonic() - start, 3)
        network['dns'].append(entry)
    network_path.write_text(json.dumps(network, ensure_ascii=False, indent=2), encoding='utf-8')
print('Network check:', network_path)

try:
    if DIRECT_PUBLIC_PROBE:
        if BROWSER_TEST_MODE != 'off':
            priority_probe.browser_preflight(crawler, RUN_DIR, user_agent=BROWSER_USER_AGENT,
                                             chrome_binary=CHROME_BINARY, load_timeout=BROWSER_LOAD_TIMEOUT)
        if RECOVERY_RUN_DIR:
            recovery = priority_probe.seed_successful_requests(crawler, RUN_DIR, SAMPLES, RECOVERY_RUN_DIR)
            print('Recovery:', len(recovery['reused_ids']), 'URL reuse raw;', len(recovery['retry_ids']), 'URL thử lại.')
        records = priority_probe.crawl_requests(
            crawler, RUN_DIR, SAMPLES, workers=WORKERS, delay=DELAY,
            attempts=PAGE_ATTEMPTS, user_agent=USER_AGENT,
            timeout=(PAGE_CONNECT_TIMEOUT, PAGE_READ_TIMEOUT), budget=PAGE_REQUEST_BUDGET, max_new=MAX_NEW)
        browser_samples = priority_probe.select_browser_samples(SAMPLES, records, BROWSER_TEST_MODE)
        if browser_samples:
            priority_probe.crawl_browser(crawler, RUN_DIR, browser_samples,
                                        user_agent=BROWSER_USER_AGENT, chrome_binary=CHROME_BINARY, delay=DELAY,
                                        load_timeout=BROWSER_LOAD_TIMEOUT, attempts=BROWSER_ATTEMPTS)
        crawler.report(argparse.Namespace(out=str(RUN_DIR)))
    else:
        crawler.locked_crawl(argparse.Namespace(
            out=str(RUN_DIR), workers=WORKERS, delay=DELAY, attempts=ATTEMPTS, max_new=MAX_NEW,
            robots_attempts=ROBOTS_ATTEMPTS, robots_connect_timeout=ROBOTS_CONNECT_TIMEOUT,
            robots_read_timeout=ROBOTS_READ_TIMEOUT, robots_retry_budget=ROBOTS_RETRY_BUDGET,
            robots_failure_ttl=ROBOTS_FAILURE_TTL,
        ))
finally:
    rows, summary, zip_path = save_outputs()


## 6. Xem lỗi riêng và tải output

`khong_vao_duoc.csv` chứa domain, URL, loại lỗi, lý do tiếng Việt và lỗi gốc.
`loi_theo_loai.csv` gom số hostname và số URL theo từng lỗi (HTTP 403, timeout,
robots trả HTML, robots cấm crawl, DNS, SSL…). File JSON trong `records/`
giữ trace request/redirect; `robots/` giữ bằng chứng robots.txt.
`network_check.json` ghi DNS của mọi hostname trong bộ mẫu để đối chiếu Kaggle/local;
`priority_probe_config.json` ghi User-Agent/header, timeout và chế độ tải trang.
`requests_vs_selenium.csv` và `browser_records/` giữ kết quả trình duyệt riêng;
`browser_setup_error.json` được ghi nếu Chrome/driver không khởi động được.
`browser_runtime.json`, `browser_preflight.json`, `chrome_dependencies.log` và `chromedriver.log`
ghi kiểm tra môi trường/cài thư viện/khởi động Chrome (log cài apt chỉ có khi cần cài thêm).
ZIP gồm kết quả, raw, ảnh chụp và snapshot để kiểm tra riêng.

Kaggle: tải ZIP trong **Output**. Colab: ô dưới hiện nút tải ZIP.
Jupyter local: dùng các liên kết file hoặc mở đường dẫn output được in ra.


In [ ]:
# Có thể chạy lại ô này để xuất báo cáo từ checkpoint mà không crawl thêm.
rows, summary, zip_path = save_outputs(show=False)
failed_rows = [r for r in rows if r['access_result'] == 'KHONG_VAO_DUOC']
print(f'Không vào được: {len(failed_rows)}/{len(rows)} URL, thuộc {len({r["domain"] for r in failed_rows})} hostname')
for row in failed_rows:
    print(f"{row['domain']} | {row['error_type']} | {row['error_detail']}")
print('\nOutput:', RUN_DIR)
for name in ['WEB_ERRORS.md', 'uu_tien_fix_domain.html', 'uu_tien_fix_domain.csv', 'tong_hop_domain.csv',
             'requests_vs_selenium.csv',
             'ket_qua.html', 'ket_qua.csv', 'vao_duoc.csv', 'khong_vao_duoc.csv',
             'loi_theo_loai.csv', 'can_xem_noi_dung.csv', 'chua_chay.csv']:
    display(FileLink(str(RUN_DIR / name)))
display(FileLink(str(zip_path)))

if 'google.colab' in sys.modules:
    import ipywidgets as widgets
    from google.colab import files
    button = widgets.Button(description='Tải ZIP kết quả')
    button.on_click(lambda _: files.download(str(zip_path)))
    display(button)
